In [1]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.13/bin/python3


In [2]:
import sys
!{sys.executable} -m pip install pytorch-crf
print("Done")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3 -m pip install --upgrade pip
Done


In [3]:
from torchcrf import CRF
print("CRF imported successfully")

CRF imported successfully


In [4]:
import sys
!{sys.executable} -m pip install transformers tokenizers seqeval pytorch-crf tf-keras


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3 -m pip install --upgrade pip


In [6]:
!pip install transformers
!pip install tokenizers
!pip install seqeval
!pip install pytorch-crf
from torchcrf import CRF
import torch
from transformers import BertModel, AutoModelForTokenClassification, AutoTokenizer, Trainer, TrainingArguments, BertConfig, get_linear_schedule_with_warmup, get_constant_schedule_with_warmup, get_cosine_with_hard_restarts_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score, accuracy_score
from tqdm import tqdm, trange
import numpy as np
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
log_soft = F.log_softmax


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
MAX_LEN = 256
bs = 32

In [8]:
def read_conll_file(file_path):
    data = []
    current_sentence = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line.startswith('-DOCSTART-'):
                continue
            if line:
                parts = line.split()
                word = parts[0]
                ner_label = parts[-1]
                current_sentence.append((word, ner_label))
            else:
                if current_sentence:
                    data.append(current_sentence)
                    current_sentence = []
    if current_sentence:
        data.append(current_sentence)
    return data

In [9]:
train_data = read_conll_file('/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/train_final.conll')
val_data = read_conll_file('/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/NER_Irish_validation.conll')
test_data = read_conll_file('/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/NER_Irish_test.conll')

In [10]:
# Finding instances of multi-word named entities

def findMWE(sentence):
    tags = [tag for _, tag in sentence]  # Extract only the tags

    # Condition 1: Sentence must contain at least one 'I-' tag
    if not any(tag.startswith('I-') for tag in tags):
        return False

    # Condition 2: Ensure all 'B-' tags are followed by an 'I-' tag
    prev_tag = None
    for tag in tags:
        if tag.startswith('B-'):
            prev_tag = tag  # Store current 'B-' tag
        elif tag.startswith('I-'):
            if prev_tag and prev_tag[2:] == tag[2:]:  # Matching entity type
                prev_tag = None  # Valid sequence, reset

    # If there's still a lingering 'B-' tag, it means it wasn't followed correctly
    return prev_tag is None


mwe_test_data = [sentence for sentence in test_data if findMWE(sentence)]
print(len(mwe_test_data))
print(mwe_test_data[:10])

89
[[('Mar', 'O'), ('a', 'O'), ('tchítear', 'O'), ('do', 'O'), ('Sheosamh', 'B-PER'), ('Mac', 'I-PER'), ('Grianna', 'I-PER'), ('é', 'O'), ('caithfidh', 'O'), ('an', 'O'), ('t-ealaíontóir', 'O'), ('an', 'O'), ('solas', 'O'), ('a', 'O'), ('thabhairt', 'O'), ('don', 'O'), ('saol', 'O'), ('agus', 'O'), ('diúltú', 'O'), ('do', 'O'), ('chathú', 'O'), ('sin', 'O'), ('na', 'O'), ('truaillíochta', 'O'), ('a', 'O'), ('chuireann', 'O'), ('an', 'O'), ('saol', 'O'), ('ina', 'O'), ('chosán', 'O'), ('.', 'O')], [('Grianghraif', 'O'), ('le', 'O'), ('Maidhc', 'B-PER'), ('Ó', 'I-PER'), ('Seachnasaí', 'I-PER'), ('.', 'O')], [('Tagann', 'O'), ('a', 'O'), ('ráiteas', 'O'), ('tar', 'O'), ('éis', 'O'), ('don', 'O'), ('chomhlacht', 'O'), ('a', 'O'), ('rá', 'O'), ('le', 'O'), ('hoibrithe', 'O'), ('an', 'O'), ('iarnróid', 'O'), ('mí', 'O'), ('ó', 'O'), ('shin', 'O'), ('go', 'O'), ('raibh', 'O'), ('sí', 'O'), ('le', 'O'), ('dúnadh', 'O'), (':', 'O'), ("'Chuamar", 'O'), ('ag', 'O'), ('cruinniú', 'O'), ('i', 'O'),

In [11]:
# Import tokenizer and create tag dictionary
tokenizer = AutoTokenizer.from_pretrained('DCU-NLP/bert-base-irish-cased-v1', do_lower_case=False)

tag_values = [
        'O',
        'B-PER',
        'I-PER',
        'B-LOC',
        'I-LOC',
        'B-ORG',
        'I-ORG'
]

tag_values.append("PAD")
tag2idx = {t: i for i, t in enumerate(tag_values)}


In [12]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [13]:
# Split the words and labels in the data
def splitWordsAndLabels(data_list):
  sent = []
  labels = []
  for lst in data_list:
    s = []
    l = []
    for words in lst:
      s.append(words[0])
      l.append(words[1])
    sent.append(s)
    labels.append(l)
  return sent, labels

train_sent, train_labels = splitWordsAndLabels(train_data)
val_sent, val_labels = splitWordsAndLabels(val_data)
test_sent, test_labels = splitWordsAndLabels(test_data)
mwe_test_sent, mwe_test_labels = splitWordsAndLabels(mwe_test_data)
print(train_sent[1])
print(train_labels[1])

['Baineann', 'mo', 'cheist', 'le', 'cúrsaí', 'tithíochta', 'i', 'nGaillimh', '-', 'tá', 'sé', 'níos', 'cirte', 'easpa', 'tithíochta', 'i', 'nGaillimh', 'a', 'rá', '-', 'agus', 'an', 'tascfhórsa', 'a', 'bunaíodh', 'breis', 'agus', 'ceithre', 'bliana', 'ó', 'shin']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [14]:
# Tokenize the data and preserve the corresponding labels/tags

def tokenize_and_preserve_labels(sentence, text_labels):
    tokenized_sentence = []
    labels = []

    for word, label in zip(sentence, text_labels):
        tokenized_word = tokenizer.tokenize(word)
        n_subwords = len(tokenized_word)
        tokenized_sentence.extend(tokenized_word)
        labels.extend([label] * n_subwords)
    return tokenized_sentence, labels

# Run the splitting function on the training, test, and validation sets
tokenized_texts_and_labels = [
    tokenize_and_preserve_labels(sent, labs)
    for sent, labs in zip(train_sent, train_labels)
]

val_tokenized_texts_and_labels = [
    tokenize_and_preserve_labels(sent, labs)
    for sent, labs in zip(val_sent, val_labels)
]

test_tokenized_texts_and_labels = [
    tokenize_and_preserve_labels(sent, labs)
    for sent, labs in zip(test_sent, test_labels)
]

mwe_test_tokenized_texts_and_labels = [
    tokenize_and_preserve_labels(sent, labs)
    for sent, labs in zip(mwe_test_sent, mwe_test_labels)
]

# Bring the sentences and labels back together after tokenizing
tokenized_texts = [token_label_pair[0] for token_label_pair in tokenized_texts_and_labels]
labels = [token_label_pair[1] for token_label_pair in tokenized_texts_and_labels]

val_tokenized_texts = [token_label_pair[0] for token_label_pair in val_tokenized_texts_and_labels]
val_labels = [token_label_pair[1] for token_label_pair in val_tokenized_texts_and_labels]

test_tokenized_texts = [token_label_pair[0] for token_label_pair in test_tokenized_texts_and_labels]
test_labels = [token_label_pair[1] for token_label_pair in test_tokenized_texts_and_labels]

mwe_test_tokenized_texts = [token_label_pair[0] for token_label_pair in mwe_test_tokenized_texts_and_labels]

print(tokenized_texts[1])
print(labels[1])

['Baineann', 'mo', 'cheist', 'le', 'cúrsaí', 'tithíochta', 'i', 'nGaillimh', '-', 'tá', 'sé', 'níos', 'cirt', '##e', 'easpa', 'tithíochta', 'i', 'nGaillimh', 'a', 'rá', '-', 'agus', 'an', 'tasc', '##fhórsa', 'a', 'bunaíodh', 'breis', 'agus', 'ceithre', 'bliana', 'ó', 'shin']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [15]:
# Get the input ids for each dataset
input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", value=0.0,
                          truncating="post", padding="post")

val_input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in val_tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", value=0.0,
                          truncating="post", padding="post")

test_input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in test_tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", value=0.0,
                          truncating="post", padding="post")

mwe_test_input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in mwe_test_tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", value=0.0,
                          truncating="post", padding="post")

# Get the tags for each dataset
tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in labels],
                     maxlen=MAX_LEN, value=tag2idx["PAD"], padding="post",
                     dtype="long", truncating="post")

val_tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in val_labels],
                     maxlen=MAX_LEN, value=tag2idx["PAD"], padding="post",
                     dtype="long", truncating="post")

test_tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in test_labels],
                     maxlen=MAX_LEN, value=tag2idx["PAD"], padding="post",
                     dtype="long", truncating="post")

mwe_test_tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in mwe_test_labels],
                     maxlen=MAX_LEN, value=tag2idx["PAD"], padding="post",
                     dtype="long", truncating="post")

# Create attention masks for each dataset
attention_masks = [[float(i != 0.0) for i in ii] for ii in input_ids]

val_attention_masks = [[float(i != 0.0) for i in ii] for ii in val_input_ids]

test_attention_masks = [[float(i != 0.0) for i in ii] for ii in test_input_ids]

mwe_test_attention_masks = [[float(i != 0.0) for i in ii] for ii in mwe_test_input_ids]

# Make the inputs, tags, and masks for each dataset into torch tensors
tr_inputs = torch.tensor(input_ids)
tr_tags = torch.tensor(tags)
tr_masks = torch.tensor(attention_masks)

vl_inputs = torch.tensor(val_input_ids)
vl_tags = torch.tensor(val_tags)
vl_masks = torch.tensor(val_attention_masks)

tst_inputs = torch.tensor(test_input_ids)
tst_tags = torch.tensor(test_tags)
tst_masks = torch.tensor(test_attention_masks)

mwe_tst_inputs = torch.tensor(mwe_test_input_ids)
mwe_tst_tags = torch.tensor(mwe_test_tags)
mwe_tst_masks = torch.tensor(mwe_test_attention_masks)

# Create a tensor dataset, sampler and dataloader for each dataset
tr_data = TensorDataset(tr_inputs, tr_masks, tr_tags)
train_sampler = RandomSampler(tr_data) # Sample from the training set randomly
train_dataloader = DataLoader(tr_data, sampler=train_sampler, batch_size=bs)

valid_data = TensorDataset(vl_inputs, vl_masks, vl_tags)
valid_sampler = SequentialSampler(valid_data) # Sample from the validation set sequentially
valid_dataloader = DataLoader(valid_data, sampler=valid_sampler, batch_size=bs)

tst_data = TensorDataset(tst_inputs, tst_masks, tst_tags)
tst_sampler = SequentialSampler(tst_data) # Sample from the test set sequentially
tst_dataloader = DataLoader(tst_data, sampler=tst_sampler, batch_size=bs)

mwe_tst_data = TensorDataset(mwe_tst_inputs, mwe_tst_masks, mwe_tst_tags)
mwe_sampler = SequentialSampler(mwe_tst_data) # Sample from the mwe test set sequentially
mwe_tst_dataloader = DataLoader(mwe_tst_data, sampler=mwe_sampler, batch_size=bs)

In [16]:
# Load the model
config = BertConfig.from_pretrained('DCU-NLP/bert-base-irish-cased-v1', output_hidden_states=True)
config.max_position_embeddings = 512

bert_model = BertModel.from_pretrained(
                        'DCU-NLP/bert-base-irish-cased-v1',
                        config=config,
                        add_pooling_layer=False
)

In [17]:
class BERT_CRF(nn.Module):
    def __init__(self, bert_model, num_labels):
        super(BERT_CRF, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.25)
        # 4 last of layer
        self.classifier = nn.Linear(4*768, num_labels)
        self.crf = CRF(num_labels, batch_first = True)

    def forward_custom(self, b_input_ids, b_input_mask,  b_labels=None, token_type_ids=None):
        outputs = self.bert(b_input_ids, attention_mask=b_input_mask)
        sequence_output = torch.cat((outputs[1][-1], outputs[1][-2], outputs[1][-3], outputs[1][-4]),-1)
        sequence_output = self.dropout(sequence_output)

        emission = self.classifier(sequence_output)

        if b_labels is not None:
            loss = -self.crf(log_soft(emission, 2), b_labels, mask=b_input_mask.type(torch.uint8), reduction='mean')
            prediction = self.crf.decode(emission, mask=b_input_mask.type(torch.uint8))
            return [loss, prediction]

        else:
            prediction = self.crf.decode(emission, mask=b_input_mask.type(torch.uint8))
            return prediction

In [18]:
model = BERT_CRF(bert_model, num_labels=len(tag2idx))
model.to(device)

BERT_CRF(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30101, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affi

In [19]:
# Optimising parameters for finetuning the model
cnt = -1
num_layer = 197
for param in model.named_parameters():
    cnt += 1
    if cnt>=num_layer:
        param[1].requires_grad = True
    else:
        param[1].requires_grad = True
    print(cnt,param[0],'\t',param[1].requires_grad)


FINETUNING = True
if FINETUNING:
    param_optimizer1 = list(model.named_parameters())[:num_layer]
    param_optimizer2 = list(model.named_parameters())[num_layer:num_layer+2]
    param_optimizer3 = list(model.named_parameters())[num_layer+2:]
    no_decay = ['bias', 'gamma', 'beta']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer1 if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 1e-5},
        {'params': [p for n, p in param_optimizer1 if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0},

        {'params': [p for n, p in param_optimizer2 if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 1e-3,
         'lr': 1e-3},
        {'params': [p for n, p in param_optimizer2 if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0,
         'lr':1e-3},

        {'params': [p for n, p in param_optimizer3 if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 1e-3,
         'lr':4e-3},
        {'params': [p for n, p in param_optimizer3 if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0,
         'lr':4e-3}
    ]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=3e-5,
    eps=1e-8
)

epochs = 10
max_grad_norm = 1.0

# Total number of training steps = number of batches * number of epochs
total_steps = len(train_dataloader) * epochs

# Defining the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps/10),
    num_training_steps=total_steps
)

0 bert.embeddings.word_embeddings.weight 	 True
1 bert.embeddings.position_embeddings.weight 	 True
2 bert.embeddings.token_type_embeddings.weight 	 True
3 bert.embeddings.LayerNorm.weight 	 True
4 bert.embeddings.LayerNorm.bias 	 True
5 bert.encoder.layer.0.attention.self.query.weight 	 True
6 bert.encoder.layer.0.attention.self.query.bias 	 True
7 bert.encoder.layer.0.attention.self.key.weight 	 True
8 bert.encoder.layer.0.attention.self.key.bias 	 True
9 bert.encoder.layer.0.attention.self.value.weight 	 True
10 bert.encoder.layer.0.attention.self.value.bias 	 True
11 bert.encoder.layer.0.attention.output.dense.weight 	 True
12 bert.encoder.layer.0.attention.output.dense.bias 	 True
13 bert.encoder.layer.0.attention.output.LayerNorm.weight 	 True
14 bert.encoder.layer.0.attention.output.LayerNorm.bias 	 True
15 bert.encoder.layer.0.intermediate.dense.weight 	 True
16 bert.encoder.layer.0.intermediate.dense.bias 	 True
17 bert.encoder.layer.0.output.dense.weight 	 True
18 bert.encode

In [20]:
## Store the average loss after each epoch so we can plot them
loss_values, validation_loss_values = [], []
best_model_state = None
patience = 2
best_val_loss = float('inf')
epochs_without_improvement = 0
best_epoch = 0

for epoch in trange(epochs, desc="Epoch"):
    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set

    # Put the model into training mode
    model.train()
    # Reset the total loss for this epoch
    total_loss = 0

    # Training loop
    for step, batch in enumerate(train_dataloader):
        # add batch to gpu
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch
        # Always clear any previously calculated gradients before performing a backward pass
        model.zero_grad()
        # forward pass
        # This will return the loss (rather than the model output)
        # because we have provided the `labels`
        outputs = model.forward_custom(b_input_ids, b_input_mask, b_labels, token_type_ids=None)
        # get the loss
        loss = outputs[0]
        # Perform a backward pass to calculate the gradients
        loss.backward()
        # track train loss
        total_loss += loss.item()
        # Clip the norm of the gradient
        # This is to help prevent the "exploding gradients" problem
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
        # update parameters
        optimizer.step()
        # Update the learning rate
        scheduler.step()

    # Calculate the average loss over the training data
    avg_train_loss = total_loss / len(train_dataloader)
    print("Average train loss: {}".format(avg_train_loss))

    # Store the loss value for plotting the learning curve
    loss_values.append(avg_train_loss)


    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set

    # Put the model into evaluation mode
    model.eval()
    # Reset the validation loss for this epoch
    eval_loss, eval_accuracy = 0, 0
    nb_eval_steps, nb_eval_examples = 0, 0
    predictions , true_labels = [], []
    for batch in valid_dataloader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch

        # Telling the model not to compute or store gradients,
        # saving memory and speeding up validation
        with torch.no_grad():
            # Forward pass, calculate logit predictions
            # This will return the logits rather than the loss because we have not provided labels
            outputs = model.forward_custom(b_input_ids, b_input_mask, b_labels, token_type_ids=None)
        # Move logits and labels to CPU
        logits = outputs[0].detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # Calculate the accuracy for this batch of test sentences
        eval_loss += outputs[0].mean().item()
        predictions.extend(outputs[1])
        true_labels.extend(label_ids)

    eval_loss = eval_loss / len(valid_dataloader)
    validation_loss_values.append(eval_loss)
    print("Validation loss: {}".format(eval_loss))
    pred_tags = []
    valid_tags = []
    for p, l in zip(predictions, true_labels):
        for p_i, l_i in zip(p, l):
            if tag_values[l_i] != "PAD":
                pred_tags.append(tag_values[p_i])
                valid_tags.append(tag_values[l_i])
                tokens = tokenizer.convert_ids_to_tokens(b_input_ids[0].to('cpu').numpy())
                new_tokens, new_labels, new_preds = [], [], []
                for token, label_idx, pred in zip(tokens, l, p):
                    if token.startswith("##"):
                        new_tokens[-1] = new_tokens[-1] + token[2:]
                    else:
                        new_labels.append(label_idx)
                        new_preds.append(pred)
                        new_tokens.append(token)
                for token, pred, label in zip(new_tokens, new_preds, new_labels):
                    pred_tags.append(tag_values[pred])
                    valid_tags.append(tag_values[label])

    print("Validation Accuracy: {}".format(accuracy_score(pred_tags, valid_tags)))
    val_report = classification_report(valid_tags, pred_tags)
    print("Epoch {} - Validation Classification Report:".format(epoch))
    print(val_report)
    if eval_loss < best_val_loss:
      best_val_loss = eval_loss
      best_epoch = epoch
      best_model_state = model.state_dict()
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1
      if epochs_without_improvement >= patience:
          print(f"Validation loss hasn't improved for {patience} epochs. Best model found at epoch {best_epoch}.")
          break

    print("Epoch {} - Training Loss: {:.4f}, Validation Loss: {:.4f}".format(epoch, avg_train_loss, eval_loss))
    print()

# Output the best epoch model
if best_model_state is not None:
    torch.save(best_model_state, "gaBERT_RDA_CRF.pt")

Epoch:   0%|                                             | 0/10 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Epoch:   0%|                                             | 0/10 [00:10<?, ?it/s]


RuntimeError: MPS backend out of memory (MPS allocated: 8.90 GiB, other allocations: 98.66 MiB, max allowed: 9.07 GiB). Tried to allocate 96.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# Training and Validation Loss Plot

# Generate x-axis values (epochs)
epochs = range(len(loss_values) + 1)

# Plotting with markers and lines
plt.plot(epochs[:-1], loss_values, marker='o', linestyle='-', label='Training Loss', color='blue')
plt.plot(epochs[:-1], validation_loss_values, marker='o', linestyle='-', label='Validation Loss', color='orange')

# Labeling and styling
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Set ticks for x-axis (epochs) to whole numbers only starting from 0
plt.xticks(epochs[:-1])


# Ensure y-axis (loss) ticks are whole numbers only
#plt.yticks(range(int(min(min(loss_values), min(validation_loss_values))), int(max(max(loss_values), max(validation_loss_values))) + 1))
plt.savefig('gabert_RDA_CRF_training_validation_loss_plot.png')
# Show plot
plt.grid(True)
plt.show()

In [ ]:
model.eval()

predicted_labels = []
true_labels = []

# Iterate through the test_dataloader to get the predictions
# Using the GPU
for batch in tst_dataloader:
    batch = tuple(t.to(device) for t in batch)
    inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'labels': batch[2]}

    # Disabling gradient calculation for evaluation
    with torch.no_grad():
        #outputs = model(**inputs)
        outputs = model.forward_custom(batch[0], batch[1], batch[2], token_type_ids=None)

    # Extracting the predictions and also the true labels for the test data
    predict_labels = outputs[1]
    #print(predict_labels)
    predictions = []
    for predict_label in predict_labels:
      #print(predict_label)
      predicted_labels.append(predict_label)
    #predicted_labels.append(predictions)

    true_labels.extend(inputs['labels'].tolist())

In [ ]:
trues = []

for sentences in true_labels:
  t = []
  for vals in sentences:
    if vals != 7:
      t.append(vals)
  trues.append(t)

preds=predicted_labels

In [ ]:
i = 0
while i < len(preds):
  if len(test_labels[i]) != len(preds[i]):
    print(f'issue: {i} {len(test_labels[i])}, {len(preds[i])}')
    print(f'{test_tokenized_texts[i]}\n{test_labels[i]}\n{preds[i]}')
    i += 1
  i += 1

In [ ]:
label_map = {
    0: 'O',
    1: 'B-PER',
    2: 'I-PER',
    3: 'B-LOC',
    4: 'I-LOC',
    5: 'B-ORG',
    6: 'I-ORG'
}

# Initialise an empty list to store DataFrames
dfs = []

# Iterate through the tokens, labels, and predictions
for tokens, labels, predictions in zip(test_tokenized_texts, test_labels, preds):
    # Create a DataFrame from the current sublist
    temp_df = pd.DataFrame({'Word': tokens, 'POS': 'X', 'True': labels, 'Predicted': [label_map[pred] for pred in predictions]})
    # Append the DataFrame to the list
    dfs.append(temp_df)
    # Add an empty row as a DataFrame to the list
    dfs.append(pd.DataFrame({'Word': [''], 'POS': [''], 'True': [''], 'Predicted': ['']}))

# Concatenate the DataFrames along the rows axis
df = pd.concat(dfs, ignore_index=True)

# Print the concatenated DataFrame
print(df)

# Check the alignment is correct
print(df.head(45))

In [ ]:
# Change to conll format for evaluation script to be run
conll_format = ""

for index, row in df.iterrows():
    text = row['Word']
    pos = row['POS']
    tag = row['True']
    mapped_tag = row['Predicted']

    # Append the token in CoNLL format (word, POS, gold_label, predicted_label)
    conll_format += f"{text}\t{pos}\t{tag}\t{mapped_tag}\n"

# Write the CoNLL format string to a text file
with open('gaBERT_RDA_CRF.conll', 'w') as f:
    f.write(conll_format)

#MWE case study

In [ ]:
model.eval()

mwe_predicted_labels = []
mwe_true_labels = []

# Iterate through the test_dataloader to get the predictions
# Using the GPU
for batch in mwe_tst_dataloader:
    batch = tuple(t.to(device) for t in batch)
    inputs = {'input_ids': batch[0], 'attention_mask': batch[1], 'labels': batch[2]}

    # Disabling gradient calculation for evaluation
    with torch.no_grad():
        #outputs = model(**inputs)
        outputs = model.forward_custom(batch[0], batch[1], batch[2], token_type_ids=None)

    # Extracting the predictions and also the true labels for the test data
    predict_labels = outputs[1]
    #print(predict_labels)
    predictions = []
    for predict_label in predict_labels:
      #print(predict_label)
      mwe_predicted_labels.append(predict_label)
    #predicted_labels.append(predictions)

    mwe_true_labels.extend(inputs['labels'].tolist())

In [ ]:
#print(mwe_predicted_labels)

In [ ]:
print(len(mwe_predicted_labels))
print(len(mwe_true_labels))

In [ ]:
print(mwe_true_labels[0])
print(mwe_predicted_labels[0])

In [ ]:
mwe_preds = []
mwe_trues = []

for sentences in mwe_true_labels:
  t = []
  for vals in sentences:
    if vals != 7:
      t.append(vals)
  mwe_trues.append(t)

for sentences in mwe_predicted_labels:
  p = []
  for vals in sentences:
    if vals != 7:
      p.append(vals)
  mwe_preds.append(p)


#print(mwe_test_tokenized_texts_and_labels)
mwe_tokenized_labels = []
for sentence, labels in mwe_test_tokenized_texts_and_labels:
    mwe_tokenized_labels.append(labels)

print(len(mwe_test_tokenized_texts[0]))
print(len(mwe_trues[0]))
print(len(mwe_tokenized_labels[0]))
print(len(mwe_preds[0]))

In [ ]:
i = 0
while i < len(mwe_tokenized_labels):
  #print(len(mwe_true_labels[i]))
  #print(len(mwe_predicted_labels[i]))
  #print(i)
  if len(mwe_tokenized_labels[i]) != len(mwe_preds[i]):
    print('issue')
    i += 1
  i += 1

In [ ]:
# Initialise an empty list to store DataFrames
dfs = []

# Iterate through the tokens, labels, and predictions
for tokens, labels, predictions in zip(mwe_test_tokenized_texts, mwe_tokenized_labels, mwe_preds):
    # Create a DataFrame from the current sublist
    temp_df = pd.DataFrame({'Word': tokens, 'POS': 'X', 'True': labels, 'Predicted': [label_map[pred] for pred in predictions]})
    # Append the DataFrame to the list
    dfs.append(temp_df)
    # Add an empty row as a DataFrame to the list
    dfs.append(pd.DataFrame({'Word': [''], 'POS': [''], 'True': [''], 'Predicted': ['']}))

# Concatenate the DataFrames along the rows axis
df = pd.concat(dfs, ignore_index=True)

# Print the concatenated DataFrame
print(df)

# Check the alignment is correct
print(df.head(45))

In [ ]:
# Change to conll format for evaluation script to be run
conll_format = ""

for index, row in df.iterrows():
    text = row['Word']
    pos = row['POS']
    tag = row['True']
    mapped_tag = row['Predicted']

    # Append the token in CoNLL format (word, POS, gold_label, predicted_label)
    conll_format += f"{text}\t{pos}\t{tag}\t{mapped_tag}\n"

# Write the CoNLL format string to a text file
with open('gaBERT_RDA_CRF_MWE.conll', 'w') as f:
    f.write(conll_format)

In [22]:
import os
base = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/'
for f in os.listdir(base):
    print(repr(f))

'Phase1_KG_Dissertation_2705.docx'
'adkins-et-al-2025-model.ipynb'
'.DS_Store'
'Phase3_KG_Dissertation_2705.docx'
'gaBERT_RDA_CRF.ipynb'
'Train:Test:Val Split'
'wikidata_cache.json'
'Switch CLAUDE 27 05.docx'
'EXTRA'
'Dissertation_2705_Phase1.ipynb'
'KG'
'Phase2_KG_Dissertation_2705.docx'
'.ipynb_checkpoints'
'venv'
'Log 27 05 Dissertation.docx'
'Dissertation_2705_Phase2.ipynb'
'adkins-et-al-2025-model_V2.ipynb'


In [25]:
import pandas as pd
import requests
import time

headers = {
    'User-Agent': 'MScDissertation/1.0 (TU Dublin; Irish NER research; michael.markey@tudublin.ie)'
}

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
loc_qids_full = pd.read_csv(KG_FILES + 'loc_qids.csv')
confident = loc_qids_full[loc_qids_full['confident'] == True].copy()
confident = confident.dropna(subset=['qid'])
confident = confident.drop_duplicates(subset=['qid'])

print(f"Fetching Logainm IDs for {len(confident)} confident LOC QIDs...")

logainm_results = []

for i, row in confident.iterrows():
    qid = row['qid']
    try:
        url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
        r = requests.get(url, timeout=10, headers=headers)
        if r.status_code == 200:
            data = r.json()
            entity = data['entities'].get(qid, {})
            claims = entity.get('claims', {})
            logainm_id = None
            if 'P6872' in claims:
                try:
                    logainm_id = claims['P6872'][0]['mainsnak']['datavalue']['value']
                except:
                    pass
            logainm_results.append({
                'qid': qid,
                'entity': row['entity'],
                'canonical': row['canonical'],
                'logainm_id': logainm_id
            })
            if logainm_id:
                print(f"  ✓ {row['canonical']} → Logainm ID: {logainm_id}")
        else:
            print(f"  ✗ {qid}: status {r.status_code}")
            logainm_results.append({
                'qid': qid,
                'entity': row['entity'],
                'canonical': row['canonical'],
                'logainm_id': None
            })
        time.sleep(0.3)
    except Exception as e:
        print(f"  ✗ {qid}: {type(e).__name__}: {e}")
        logainm_results.append({
            'qid': qid,
            'entity': row['entity'],
            'canonical': row['canonical'],
            'logainm_id': None
        })

df_logainm = pd.DataFrame(logainm_results)
found = df_logainm[df_logainm['logainm_id'].notna()]
print(f"\nLogainm IDs found: {len(found)} / {len(confident)}")
print(found.head(10).to_string())
df_logainm.to_csv(KG_FILES + 'loc_logainm_ids.csv', index=False)
print("Saved to loc_logainm_ids.csv")

Fetching Logainm IDs for 176 confident LOC QIDs...

Logainm IDs found: 0 / 176
Empty DataFrame
Columns: [qid, entity, canonical, logainm_id]
Index: []
Saved to loc_logainm_ids.csv


In [26]:
import pandas as pd
import requests
import time
import json

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
LOGAINM_API_KEY = "your_key_here"

HEADERS_WD = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"
}
HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

loc_qids_full = pd.read_csv(KG_FILES + 'loc_qids.csv')
confident = loc_qids_full[loc_qids_full['confident'] == True].copy()
confident = confident.dropna(subset=['qid']).drop_duplicates(subset=['qid'])
print(f"Processing {len(confident)} confident LOC QIDs...")

results = []

for i, row in confident.iterrows():
    qid = row['qid']
    canonical = row['canonical']
    result = {
        'qid': qid,
        'entity': row['entity'],
        'canonical': canonical,
        'logainm_id': None,
        'name_ga': None,
        'name_en': None,
        'genitive': None,
        'county_ga': None,
        'gaeltacht': False,
        'source': None
    }

    # Step 1: try to get Logainm ID from Wikidata P6872
    try:
        url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
        r = requests.get(url, headers=HEADERS_WD, timeout=10)
        if r.status_code == 200:
            data = r.json()
            claims = data['entities'].get(qid, {}).get('claims', {})
            if 'P6872' in claims:
                logainm_id = claims['P6872'][0]['mainsnak']['datavalue']['value']
                result['logainm_id'] = logainm_id
                result['source'] = 'wikidata_p6872'
        time.sleep(0.3)
    except Exception as e:
        print(f"  WD error {qid}: {e}")

    # Step 2: if we have a Logainm ID, fetch from Logainm directly
    if result['logainm_id']:
        try:
            r2 = requests.get(
                f"https://www.logainm.ie/api/v1.0/{result['logainm_id']}",
                headers=HEADERS_LG, timeout=10
            )
            if r2.status_code == 200:
                data2 = r2.json()
                for pn in data2.get('placenames', []):
                    if pn['language'] == 'ga' and pn['main']:
                        result['name_ga'] = pn['wording']
                        result['genitive'] = pn.get('genitive')
                    if pn['language'] == 'en' and pn['main']:
                        result['name_en'] = pn['wording']
                for parent in data2.get('includedIn', []):
                    if parent.get('category', {}).get('id') == 'CON':
                        result['county_ga'] = parent.get('nameGA')
                result['gaeltacht'] = data2.get('gaeltacht') is not None
                print(f"  ✓ WD→LG: {canonical} → {result['name_ga']} (genitive: {result['genitive']})")
            time.sleep(0.4)
        except Exception as e:
            print(f"  LG error {result['logainm_id']}: {e}")

    results.append(result)

df = pd.DataFrame(results)
found_wikidata = df['logainm_id'].notna().sum()
found_logainm  = df['name_ga'].notna().sum()

print(f"\n=== RESULTS ===")
print(f"Logainm IDs via Wikidata P6872: {found_wikidata}/{len(confident)}")
print(f"Full Logainm records fetched:   {found_logainm}/{len(confident)}")
print(f"\nSample with genitive forms:")
print(df[df['genitive'].notna()][['canonical','name_ga','genitive','county_ga']].head(10).to_string(index=False))

df.to_csv(KG_FILES + 'loc_logainm_enriched.csv', index=False)
print(f"\nSaved → {KG_FILES}loc_logainm_enriched.csv")

Processing 176 confident LOC QIDs...

=== RESULTS ===
Logainm IDs via Wikidata P6872: 0/176
Full Logainm records fetched:   0/176

Sample with genitive forms:
Empty DataFrame
Columns: [canonical, name_ga, genitive, county_ga]
Index: []

Saved → /Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/loc_logainm_enriched.csv


In [27]:
import pandas as pd
import requests
import time

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
LOGAINM_API_KEY = "your_key_here"

HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

loc_qids_full = pd.read_csv(KG_FILES + 'loc_qids.csv')
confident = loc_qids_full[loc_qids_full['confident'] == True].copy()
confident = confident.dropna(subset=['qid']).drop_duplicates(subset=['qid'])

# Also load loc_entities for the surface forms
loc_entities = pd.read_csv('/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/loc_entities.csv')

print(f"Processing {len(confident)} LOC entities via Logainm text search...")

results = []

for i, row in confident.iterrows():
    canonical = str(row['canonical']).strip()
    result = {
        'qid': row['qid'],
        'entity': row['entity'],
        'canonical': canonical,
        'logainm_id': None,
        'name_ga': None,
        'name_en': None,
        'genitive': None,
        'county_ga': None,
        'gaeltacht': False,
    }

    # Try canonical form first, then strip leading articles
    search_terms = [canonical]
    for prefix in ['An ', 'Na ', 'an ', 'na ']:
        if canonical.startswith(prefix):
            search_terms.append(canonical[len(prefix):])

    for term in search_terms:
        try:
            r = requests.get(
                "https://www.logainm.ie/api/v1.0/",
                headers=HEADERS_LG,
                params={"Query": term, "PerPage": 1},
                timeout=30  # much longer timeout
            )
            if r.status_code == 200:
                data = r.json()
                hits = data.get('results', [])
                if hits:
                    top = hits[0]
                    result['logainm_id'] = top.get('id')
                    for pn in top.get('placenames', []):
                        if pn['language'] == 'ga' and pn['main']:
                            result['name_ga'] = pn['wording']
                            result['genitive'] = pn.get('genitive')
                        if pn['language'] == 'en' and pn['main']:
                            result['name_en'] = pn['wording']
                    for parent in top.get('includedIn', []):
                        if parent.get('category', {}).get('id') == 'CON':
                            result['county_ga'] = parent.get('nameGA')
                    result['gaeltacht'] = top.get('gaeltacht') is not None
                    print(f"  ✓ [{i+1}/{len(confident)}] {canonical} → {result['name_ga']} (gen: {result['genitive']})")
                    break  # found a result, stop trying other terms
            time.sleep(0.5)
        except requests.exceptions.Timeout:
            print(f"  ⏱ [{i+1}/{len(confident)}] {term} — timeout, skipping")
            break
        except Exception as e:
            print(f"  ✗ [{i+1}/{len(confident)}] {term} — {type(e).__name__}: {e}")
            break

    if not result['logainm_id']:
        print(f"  ✗ [{i+1}/{len(confident)}] {canonical} — not found")

    results.append(result)

df = pd.DataFrame(results)
found = df['logainm_id'].notna().sum()
gaeltacht = df[df['gaeltacht'] == True].shape[0]

print(f"\n=== RESULTS ===")
print(f"Found in Logainm:  {found}/{len(confident)}")
print(f"Gaeltacht places:  {gaeltacht}")
print(f"\nWith genitive forms:")
print(df[df['genitive'].notna()][['canonical','name_ga','genitive','county_ga']].head(15).to_string(index=False))

df.to_csv(KG_FILES + 'loc_logainm_enriched.csv', index=False)
print(f"\nSaved → {KG_FILES}loc_logainm_enriched.csv")

Processing 176 LOC entities via Logainm text search...
  ✗ [1/176] Gaeltacht — not found
  ✗ [2/176] Baile Átha Cliath — not found
  ✗ [5/176] Ceathrú Rua — not found
  ✗ [7/176] Gaeltachta — not found
  ✗ [9/176] Conamara — not found
  ✗ [13/176] Inis Meáin — not found
  ✗ [15/176] Gaoth Dobhair — not found
  ✗ [17/176] Gaillimh — not found
  ✗ [18/176] Port Omna — not found
  ✗ [22/176] Maigh Eo — not found
  ✗ [24/176] Corcaigh — not found
  ✗ [25/176] Sráid Henrietta — not found
  ✗ [26/176] 14 Sráid Henrietta — not found
  ✗ [31/176] Baile Átha Luain — not found
  ✗ [32/176] Spidéal — not found
  ✗ [34/176] Cill Dara — not found
  ✗ [37/176] Ard-Mhúsaem na hÉireann — not found
  ✗ [38/176] Inis Oírr — not found
  ✗ [39/176] Ciarraí — not found
  ✗ [41/176] Tír Chonaill — not found
  ✗ [42/176] Dún Chaoin — not found
  ✗ [46/176] Contae an Chláir — not found
  ✗ [48/176] Lios Tuathail — not found
  ✗ [54/176] Port Láirge — not found
  ✗ [57/176] cathair Bhaile Átha Cliath — not fou

In [31]:
import requests
import json

LOGAINM_API_KEY = "IFqzIyDVlVIU89So2tn6TXvb2LHGAl"
HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

# Test: fetch first page of Galway places
r = requests.get(
    "https://www.logainm.ie/api/v1.0/",
    headers=HEADERS_LG,
    params={
        "PlaceID": 100015,  # County Galway
        "ExcludeStreets": True,
        "PerPage": 5,
        "Page": 1
    },
    timeout=30
)

print(f"Status: {r.status_code}")
data = r.json()
print(f"Total results: {data.get('count', 'N/A')}")
print(f"Pages: {data.get('pageCount', 'N/A')}")
print(json.dumps(data.get('results', [])[:2], indent=2, ensure_ascii=False))

Status: 200
Total results: N/A
Pages: N/A
[
  {
    "id": 68,
    "dateCreated": "2007-07-27T10:34:30.31",
    "dateModified": "2025-01-10T14:00:35.06",
    "permalink": "https://www.logainm.ie/68.aspx",
    "featured": [],
    "cluster": {
      "focusID": 217,
      "members": [
        {
          "placeID": 217,
          "category": {
            "id": "BAR",
            "nameEN": "barony",
            "nameGA": "barúntacht"
          }
        },
        {
          "placeID": 68,
          "category": {
            "id": "BAR",
            "nameEN": "barony",
            "nameGA": "barúntacht"
          }
        }
      ]
    },
    "placenames": [
      {
        "id": 808939,
        "language": "ga",
        "wording": "Maigh Charnáin",
        "genitive": "Mhaigh Charnáin",
        "main": true,
        "acceptability": {
          "id": 8,
          "textEN": "non-validated name",
          "textGA": "ainm neamhdheimhnithe"
        },
        "audio": {
          "fileName

In [29]:
r = requests.get(
    "https://www.logainm.ie/api/v1.0/",
    headers=HEADERS_LG,
    params={"PlaceID": 100015, "ExcludeStreets": True, "PerPage": 5},
    timeout=30
)
print(f"Status: {r.status_code}")
print(f"Response text: {r.text[:500]}")

Status: 401
Response text: 


In [30]:
r2 = requests.get(
    "https://www.logainm.ie/api/v1.0/1166137",
    headers=HEADERS_LG,
    timeout=10
)
print(f"Direct ID status: {r2.status_code}")

Direct ID status: 401


In [33]:
import requests
import json
import time
import pandas as pd
from pathlib import Path

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
LOGAINM_API_KEY = "IFqzIyDVlVIU89So2tn6TXvb2LHGAl"
HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

# All 26 county PlaceIDs
COUNTIES = {
    "Antrim": 100001, "Armagh": 100002, "Carlow": 100003, "Cavan": 100004,
    "Clare": 100005, "Cork": 100006, "Derry": 100007, "Donegal": 100008,
    "Down": 100009, "Dublin": 100010, "Fermanagh": 100011, "Galway": 100012,
    "Kerry": 100013, "Kildare": 100014, "Kilkenny": 100015, "Laois": 100016,
    "Leitrim": 100017, "Limerick": 100018, "Longford": 100019, "Louth": 100020,
    "Mayo": 100021, "Meath": 100022, "Monaghan": 100023, "Offaly": 100024,
    "Roscommon": 100025, "Sligo": 100026, "Tipperary": 100027, "Tyrone": 100028,
    "Waterford": 100029, "Westmeath": 100030, "Wexford": 100031, "Wicklow": 100032
}

def fetch_county_places(county_name, county_id):
    """Fetch all places in a county, paginating through results."""
    all_places = []
    page = 1
    while True:
        try:
            r = requests.get(
                "https://www.logainm.ie/api/v1.0/",
                headers=HEADERS_LG,
                params={
                    "PlaceID": county_id,
                    "ExcludeStreets": True,
                    "PerPage": 1000,
                    "Page": page
                },
                timeout=60
            )
            if r.status_code != 200:
                print(f"  {county_name} page {page}: status {r.status_code}")
                break
            data = r.json()
            results = data.get('results', [])
            if not results:
                break
            all_places.extend(results)
            total = data.get('totalCount', 0)
            print(f"  {county_name}: page {page}, got {len(results)} places (total so far: {len(all_places)}/{total})")
            if len(all_places) >= total:
                break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"  {county_name} page {page} error: {e}")
            break
    return all_places

# Build the full lookup table
all_places = []
for county_name, county_id in COUNTIES.items():
    print(f"\nFetching {county_name}...")
    places = fetch_county_places(county_name, county_id)
    all_places.extend(places)
    time.sleep(0.5)

print(f"\nTotal places fetched: {len(all_places)}")

# Build lookup: name_ga → record, name_en → record
lookup_ga = {}
lookup_en = {}
lookup_genitive = {}

for place in all_places:
    record = {
        'logainm_id': place.get('id'),
        'name_ga': None,
        'name_en': None,
        'genitive': None,
        'county_ga': None,
        'gaeltacht': place.get('gaeltacht') is not None,
        'category': place.get('categories', [{}])[0].get('nameEN') if place.get('categories') else None
    }
    for pn in place.get('placenames', []):
        if pn['language'] == 'ga' and pn['main']:
            record['name_ga'] = pn['wording']
            record['genitive'] = pn.get('genitive')
        if pn['language'] == 'en' and pn['main']:
            record['name_en'] = pn['wording']
    for parent in place.get('includedIn', []):
        if parent.get('category', {}).get('id') == 'CON':
            record['county_ga'] = parent.get('nameGA')

    if record['name_ga']:
        lookup_ga[record['name_ga'].strip()] = record
        if record['genitive']:
            lookup_genitive[record['genitive'].strip()] = record
    if record['name_en']:
        lookup_en[record['name_en'].strip().lower()] = record

print(f"Lookup table: {len(lookup_ga)} Irish entries, {len(lookup_en)} English entries")

# Save the full lookup for reuse
import pickle
with open(KG_FILES + 'logainm_lookup.pkl', 'wb') as f:
    pickle.dump({'ga': lookup_ga, 'en': lookup_en, 'genitive': lookup_genitive}, f)
print(f"Saved lookup → {KG_FILES}logainm_lookup.pkl")


Fetching Antrim...
  Antrim: page 1, got 1000 places (total so far: 1000/1126)
  Antrim: page 2, got 126 places (total so far: 1126/1126)

Fetching Armagh...
  Armagh: page 1, got 1000 places (total so far: 1000/2192)
  Armagh: page 2, got 1000 places (total so far: 2000/2192)
  Armagh: page 3, got 192 places (total so far: 2192/2192)

Fetching Carlow...
  Carlow: page 1, got 1000 places (total so far: 1000/2513)
  Carlow: page 2, got 1000 places (total so far: 2000/2513)
  Carlow: page 3, got 513 places (total so far: 2513/2513)

Fetching Cavan...
  Cavan: page 1, got 827 places (total so far: 827/827)

Fetching Clare...
  Clare: page 1, got 1000 places (total so far: 1000/5954)
  Clare: page 2, got 1000 places (total so far: 2000/5954)
  Clare: page 3, got 1000 places (total so far: 3000/5954)
  Clare: page 4, got 1000 places (total so far: 4000/5954)
  Clare: page 5, got 1000 places (total so far: 5000/5954)
  Clare: page 6, got 954 places (total so far: 5954/5954)

Fetching Cork..

In [35]:
import requests
import time
import pickle

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
LOGAINM_API_KEY = "0eB91Ffczl8ZC5BfVuhPVlcFe4pe4l"
HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

MISSING_COUNTIES = {
    "Kerry":    (100013, 3),   # resume from page 4
    "Kilkenny": (100015, 1),   # start from page 1
    "Wicklow":  (100032, 1),   # start from page 1
}

def fetch_county_from_page(county_name, county_id, start_page):
    all_places = []
    page = start_page
    while True:
        try:
            r = requests.get(
                "https://www.logainm.ie/api/v1.0/",
                headers=HEADERS_LG,
                params={
                    "PlaceID": county_id,
                    "ExcludeStreets": True,
                    "PerPage": 1000,
                    "Page": page
                },
                timeout=60
            )
            if r.status_code != 200:
                print(f"  {county_name} page {page}: status {r.status_code}, stopping")
                break
            data = r.json()
            results = data.get('results', [])
            if not results:
                print(f"  {county_name} page {page}: no results, done")
                break
            all_places.extend(results)
            total = data.get('totalCount', 0)
            print(f"  {county_name}: page {page}, got {len(results)} (total so far: {len(all_places)})")
            if len(all_places) >= (total - (start_page-1)*1000):
                break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"  {county_name} page {page} error: {e}, retrying in 5s...")
            time.sleep(5)
            continue  # retry same page
    return all_places

# Load existing lookup
with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)

lookup_ga       = lookup['ga']
lookup_en       = lookup['en']
lookup_genitive = lookup['genitive']

print(f"Existing lookup: {len(lookup_ga)} Irish, {len(lookup_en)} English entries")

# Fetch missing counties and add to lookup
new_places = []
for county_name, (county_id, start_page) in MISSING_COUNTIES.items():
    print(f"\nFetching {county_name} from page {start_page}...")
    places = fetch_county_from_page(county_name, county_id, start_page)
    new_places.extend(places)
    print(f"  {county_name}: fetched {len(places)} additional places")
    time.sleep(1)

print(f"\nTotal new places: {len(new_places)}")

# Add new places to lookup
added = 0
for place in new_places:
    record = {
        'logainm_id': place.get('id'),
        'name_ga': None,
        'name_en': None,
        'genitive': None,
        'county_ga': None,
        'gaeltacht': place.get('gaeltacht') is not None,
        'category': place.get('categories', [{}])[0].get('nameEN') if place.get('categories') else None
    }
    for pn in place.get('placenames', []):
        if pn['language'] == 'ga' and pn['main']:
            record['name_ga'] = pn['wording']
            record['genitive'] = pn.get('genitive')
        if pn['language'] == 'en' and pn['main']:
            record['name_en'] = pn['wording']
    for parent in place.get('includedIn', []):
        if parent.get('category', {}).get('id') == 'CON':
            record['county_ga'] = parent.get('nameGA')

    if record['name_ga'] and record['name_ga'] not in lookup_ga:
        lookup_ga[record['name_ga'].strip()] = record
        added += 1
    if record['genitive'] and record['genitive'] not in lookup_genitive:
        lookup_genitive[record['genitive'].strip()] = record
    if record['name_en'] and record['name_en'].lower() not in lookup_en:
        lookup_en[record['name_en'].strip().lower()] = record

print(f"New entries added to lookup: {added}")
print(f"Updated lookup: {len(lookup_ga)} Irish, {len(lookup_en)} English entries")

# Save updated lookup
with open(KG_FILES + 'logainm_lookup.pkl', 'wb') as f:
    pickle.dump({'ga': lookup_ga, 'en': lookup_en, 'genitive': lookup_genitive}, f)
print(f"Saved updated lookup → {KG_FILES}logainm_lookup.pkl")

Existing lookup: 38434 Irish, 51885 English entries

Fetching Kerry from page 3...
  Kerry: page 3, got 1000 (total so far: 1000)
  Kerry: page 4, got 1000 (total so far: 2000)
  Kerry: page 5, got 1000 (total so far: 3000)
  Kerry: page 6, got 931 (total so far: 3931)
  Kerry: fetched 3931 additional places

Fetching Kilkenny from page 1...
  Kilkenny: page 1, got 1000 (total so far: 1000)
  Kilkenny: page 2, got 1000 (total so far: 2000)
  Kilkenny: page 3, got 1000 (total so far: 3000)
  Kilkenny: page 4, got 1000 (total so far: 4000)
  Kilkenny: page 5, got 1000 (total so far: 5000)
  Kilkenny: page 6, got 1000 (total so far: 6000)
  Kilkenny: page 7, got 1000 (total so far: 7000)
  Kilkenny: page 8, got 519 (total so far: 7519)
  Kilkenny: fetched 7519 additional places

Fetching Wicklow from page 1...
  Wicklow page 1: no results, done
  Wicklow: fetched 0 additional places

Total new places: 11450
New entries added to lookup: 5828
Updated lookup: 44262 Irish, 58880 English entri

In [36]:
r = requests.get(
    "https://www.logainm.ie/api/v1.0/counties/",
    headers=HEADERS_LG,
    timeout=30
)
print(f"Status: {r.status_code}")
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

Status: 200
{
  "totalCount": 32,
  "totalPages": 1,
  "currentPage": 1,
  "countPerPage": 1000,
  "results": [
    {
      "id": 100000,
      "dateCreated": "2007-07-27T10:34:28.53",
      "dateModified": "2021-05-26T11:41:00.423",
      "permalink": "https://www.logainm.ie/100000.aspx",
      "featured": [],
      "cluster": {
        "focusID": 100000,
        "members": [
          {
            "placeID": 100000,
            "category": {
              "id": "CON",
              "nameEN": "county",
              "nameGA": "contae"
            }
          },
          {
            "placeID": 61266,
            "category": {
              "id": "PAR",
              "nameEN": "civil parish",
              "nameGA": "paróiste dlí"
            }
          },
          {
            "placeID": 1411588,
            "category": {
              "id": "B",
              "nameEN": "town",
              "nameGA": "baile"
            }
          }
        ]
      },
      "placenames": [
   

In [37]:
import requests
import time
import pickle

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
LOGAINM_API_KEY = "your_key_here"
HEADERS_LG = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

# Correct IDs from the counties endpoint
CORRECT_COUNTIES = {
    "Wicklow": 100008,   # was missing entirely
}

def fetch_county_from_page(county_name, county_id, start_page=1):
    all_places = []
    page = start_page
    while True:
        try:
            r = requests.get(
                "https://www.logainm.ie/api/v1.0/",
                headers=HEADERS_LG,
                params={"PlaceID": county_id, "ExcludeStreets": True, "PerPage": 1000, "Page": page},
                timeout=60
            )
            if r.status_code != 200:
                print(f"  {county_name} page {page}: status {r.status_code}")
                break
            data = r.json()
            results = data.get('results', [])
            if not results:
                print(f"  {county_name} page {page}: no results, done")
                break
            all_places.extend(results)
            total = data.get('totalCount', 0)
            print(f"  {county_name}: page {page}, got {len(results)} (total: {len(all_places)}/{total})")
            if len(all_places) >= total:
                break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"  {county_name} page {page} error: {e}, retrying...")
            time.sleep(5)
            continue
    return all_places

# Load existing lookup
with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)
lookup_ga = lookup['ga']
lookup_en = lookup['en']
lookup_genitive = lookup['genitive']
print(f"Existing: {len(lookup_ga)} Irish, {len(lookup_en)} English entries")

# Fetch Wicklow
new_places = []
for county_name, county_id in CORRECT_COUNTIES.items():
    print(f"\nFetching {county_name} (ID: {county_id})...")
    places = fetch_county_from_page(county_name, county_id)
    new_places.extend(places)
    print(f"  Got {len(places)} places")

# Add to lookup
added = 0
for place in new_places:
    record = {
        'logainm_id': place.get('id'),
        'name_ga': None, 'name_en': None, 'genitive': None,
        'county_ga': None,
        'gaeltacht': place.get('gaeltacht') is not None,
        'category': place.get('categories', [{}])[0].get('nameEN') if place.get('categories') else None
    }
    for pn in place.get('placenames', []):
        if pn['language'] == 'ga' and pn['main']:
            record['name_ga'] = pn['wording']
            record['genitive'] = pn.get('genitive')
        if pn['language'] == 'en' and pn['main']:
            record['name_en'] = pn['wording']
    for parent in place.get('includedIn', []):
        if parent.get('category', {}).get('id') == 'CON':
            record['county_ga'] = parent.get('nameGA')
    if record['name_ga'] and record['name_ga'] not in lookup_ga:
        lookup_ga[record['name_ga'].strip()] = record
        added += 1
    if record['genitive'] and record['genitive'] not in lookup_genitive:
        lookup_genitive[record['genitive'].strip()] = record
    if record['name_en'] and record['name_en'].lower() not in lookup_en:
        lookup_en[record['name_en'].strip().lower()] = record

print(f"\nAdded {added} new entries")
print(f"Final lookup: {len(lookup_ga)} Irish, {len(lookup_en)} English entries")

with open(KG_FILES + 'logainm_lookup.pkl', 'wb') as f:
    pickle.dump({'ga': lookup_ga, 'en': lookup_en, 'genitive': lookup_genitive}, f)
print("Saved.")

Existing: 44262 Irish, 58880 English entries

Fetching Wicklow (ID: 100008)...
  Wicklow page 1: status 401
  Got 0 places

Added 0 new entries
Final lookup: 44262 Irish, 58880 English entries
Saved.


In [39]:
import pandas as pd
import pickle

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'

with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)

lookup_ga       = lookup['ga']
lookup_en       = lookup['en']
lookup_genitive = lookup['genitive']

print(f"Lookup: {len(lookup_ga)} Irish, {len(lookup_en)} English, {len(lookup_genitive)} genitive entries")

loc_qids = pd.read_csv(KG_FILES + 'loc_qids.csv')
confident = loc_qids[loc_qids['confident'] == True].dropna(subset=['qid']).drop_duplicates(subset=['qid'])

def strip_mutations(text):
    text = text.strip()
    for prefix in ['An t-', 'An T-', 'An t', 'An T', 'Na h-', 'Na H-',
                   'Na h', 'Na H', 'An ', 'Na ', 'an ', 'na ']:
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    eclipsis = [('mb','b'),('gc','c'),('nd','d'),('bhf','f'),('ng','g'),
                ('bp','p'),('dt','t'),('mB','B'),('gC','C'),('nD','D'),
                ('bhF','F'),('nG','G'),('bP','P'),('dT','T')]
    for mut, base in eclipsis:
        if text.startswith(mut):
            text = base + text[len(mut):]
            break
    lenited = ['Bh','Ch','Dh','Fh','Gh','Mh','Ph','Sh','Th',
               'bh','ch','dh','fh','gh','mh','ph','sh','th']
    for l in lenited:
        if text.startswith(l):
            text = l[0].upper() + text[len(l):]
            break
    return text.strip()

results = []
for _, row in confident.iterrows():
    canonical = str(row['canonical']).strip()
    match = None
    match_type = None

    if canonical in lookup_ga:
        match = lookup_ga[canonical]; match_type = 'exact_ga'
    elif canonical.lower() in lookup_en:
        match = lookup_en[canonical.lower()]; match_type = 'exact_en'
    elif canonical in lookup_genitive:
        match = lookup_genitive[canonical]; match_type = 'genitive'
    else:
        stripped = strip_mutations(canonical)
        if stripped in lookup_ga:
            match = lookup_ga[stripped]; match_type = 'stripped_ga'
        elif stripped.lower() in lookup_en:
            match = lookup_en[stripped.lower()]; match_type = 'stripped_en'
        elif stripped in lookup_genitive:
            match = lookup_genitive[stripped]; match_type = 'stripped_genitive'

    if match:
        results.append({
            'qid': row['qid'], 'entity': row['entity'], 'canonical': canonical,
            'match_type': match_type, 'logainm_id': match['logainm_id'],
            'name_ga': match['name_ga'], 'name_en': match['name_en'],
            'genitive': match['genitive'], 'county_ga': match['county_ga'],
            'gaeltacht': match['gaeltacht'], 'category': match['category']
        })
        print(f"  ✓ [{match_type}] {canonical} → {match['name_ga']} (gen: {match['genitive']})")
    else:
        results.append({
            'qid': row['qid'], 'entity': row['entity'], 'canonical': canonical,
            'match_type': None, 'logainm_id': None, 'name_ga': None,
            'name_en': None, 'genitive': None, 'county_ga': None,
            'gaeltacht': False, 'category': None
        })
        print(f"  ✗ {canonical}")

df = pd.DataFrame(results)
matched = df['logainm_id'].notna()

print(f"\n=== MATCHING RESULTS ===")
print(f"Total LOC entities:     {len(df)}")
print(f"Matched in Logainm:     {matched.sum()} ({100*matched.sum()//len(df)}%)")
print(f"Gaeltacht places:       {df[matched]['gaeltacht'].sum()}")
print(f"\nMatch type breakdown:")
print(df[matched]['match_type'].value_counts().to_string())
print(f"\nSample matched with genitive:")
print(df[df['genitive'].notna()][['canonical','name_ga','genitive','county_ga']].head(15).to_string(index=False))
print(f"\nUnmatched (sample):")
print(df[~matched]['canonical'].head(20).tolist())

df.to_csv(KG_FILES + 'loc_logainm_enriched.csv', index=False)
print(f"\nSaved → {KG_FILES}loc_logainm_enriched.csv")

Lookup: 44262 Irish, 58880 English, 43550 genitive entries
  ✗ Gaeltacht
  ✓ [exact_ga] Baile Átha Cliath → Baile Átha Cliath (gen: Bhaile Átha Cliath)
  ✓ [exact_ga] Ceathrú Rua → Ceathrú Rua (gen: Cheathrú Rua)
  ✗ Gaeltachta
  ✓ [exact_ga] Conamara → Conamara (gen: Chonamara)
  ✓ [exact_ga] Inis Meáin → Inis Meáin (gen: Inis Meáin)
  ✓ [exact_ga] Gaoth Dobhair → Gaoth Dobhair (gen: Ghaoth Dobhair)
  ✓ [exact_ga] Gaillimh → Gaillimh (gen: na Gaillimhe)
  ✓ [exact_ga] Port Omna → Port Omna (gen: Phort Omna)
  ✓ [exact_ga] Maigh Eo → Maigh Eo (gen: Mhaigh Eo)
  ✓ [exact_ga] Corcaigh → Corcaigh (gen: Chorcaí)
  ✗ Sráid Henrietta
  ✗ 14 Sráid Henrietta
  ✓ [exact_ga] Baile Átha Luain → Baile Átha Luain (gen: Bhaile Átha Luain)
  ✗ Spidéal
  ✓ [exact_ga] Cill Dara → Cill Dara (gen: Chill Dara)
  ✗ Ard-Mhúsaem na hÉireann
  ✓ [exact_ga] Inis Oírr → Inis Oírr (gen: Inis Oírr)
  ✗ Ciarraí
  ✓ [exact_ga] Tír Chonaill → Tír Chonaill (gen: Thír Chonaill)
  ✓ [exact_ga] Dún Chaoin → Dún Chaoin (

In [40]:
import requests

# Try both keys
KEYS = {
    "original": "IFqzIyDVlVIU89So2tn6TXvb2LHGAl",
    "newer":    "0eB91Ffczl8ZC5BfVuhPVlcFe4pe4l"
}

for label, key in KEYS.items():
    r = requests.get(
        "https://www.logainm.ie/api/v1.0/",
        headers={
            "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
            "X-Api-Key": key
        },
        params={"PlaceID": 100032, "ExcludeStreets": True, "PerPage": 10, "Page": 1},
        timeout=30
    )
    print(f"{label} key → status {r.status_code}, results: {len(r.json().get('results', []))}")

original key → status 200, results: 0
newer key → status 200, results: 0


In [41]:
import requests

KEY = "IFqzIyDVlVIU89So2tn6TXvb2LHGAl"
HEADERS = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": KEY
}

# 1. Check what PlaceID 100032 actually is
r = requests.get(
    "https://www.logainm.ie/api/v1.0/100032",
    headers=HEADERS,
    timeout=30
)
print("PlaceID 100032 lookup:")
data = r.json()
print(f"  Status: {r.status_code}")
print(f"  Names: {[pn.get('wording') for pn in data.get('placenames', [])]}")
print(f"  Category: {data.get('categories')}")
print()

# 2. Try WITHOUT ExcludeStreets
r2 = requests.get(
    "https://www.logainm.ie/api/v1.0/",
    headers=HEADERS,
    params={"PlaceID": 100032, "PerPage": 10, "Page": 1},
    timeout=30
)
data2 = r2.json()
print(f"Without ExcludeStreets → totalCount: {data2.get('totalCount')}, results: {len(data2.get('results', []))}")

# 3. Try searching by name to find the real Wicklow ID
r3 = requests.get(
    "https://www.logainm.ie/api/v1.0/",
    headers=HEADERS,
    params={"query": "Wicklow", "PerPage": 5},
    timeout=30
)
data3 = r3.json()
print(f"\nSearch 'Wicklow' → {data3.get('totalCount')} results")
for res in data3.get('results', [])[:5]:
    names = [pn.get('wording') for pn in res.get('placenames', [])]
    cats = [c.get('nameEN') for c in res.get('categories', [])]
    print(f"  ID: {res.get('id')}, Names: {names}, Category: {cats}")

PlaceID 100032 lookup:
  Status: 404
  Names: []
  Category: None

Without ExcludeStreets → totalCount: 0, results: 0

Search 'Wicklow' → 4 results
  ID: 55959, Names: ['Cill Mhantáin', 'Wicklow'], Category: ['town']
  ID: 100008, Names: ['Cill Mhantáin', 'Wicklow'], Category: ['county']
  ID: 1411706, Names: ['Cill Mhantáin', 'Wicklow'], Category: ['townland']
  ID: 1437822, Names: ['Cill Mhantáin', 'Wicklow'], Category: ['municipal district']


In [42]:
import requests, pickle, time

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
KEY = "IFqzIyDVlVIU89So2tn6TXvb2LHGAl"
HEADERS = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": KEY
}

# Fetch Wicklow (correct ID: 100008)
all_places = []
page = 1
while True:
    r = requests.get(
        "https://www.logainm.ie/api/v1.0/",
        headers=HEADERS,
        params={"PlaceID": 100008, "ExcludeStreets": True, "PerPage": 1000, "Page": page},
        timeout=60
    )
    data = r.json()
    results = data.get('results', [])
    if not results:
        break
    all_places.extend(results)
    total = data.get('totalCount', 0)
    print(f"Page {page}: {len(results)} results (total so far: {len(all_places)}/{total})")
    if len(all_places) >= total:
        break
    page += 1
    time.sleep(0.5)

print(f"\nFetched {len(all_places)} Wicklow places")

# Load and update lookup
with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)

lookup_ga, lookup_en, lookup_genitive = lookup['ga'], lookup['en'], lookup['genitive']

added = 0
for place in all_places:
    record = {
        'logainm_id': place.get('id'),
        'name_ga': None, 'name_en': None, 'genitive': None,
        'county_ga': None,
        'gaeltacht': place.get('gaeltacht') is not None,
        'category': place.get('categories', [{}])[0].get('nameEN') if place.get('categories') else None
    }
    for pn in place.get('placenames', []):
        if pn['language'] == 'ga' and pn['main']:
            record['name_ga'] = pn['wording']
            record['genitive'] = pn.get('genitive')
        if pn['language'] == 'en' and pn['main']:
            record['name_en'] = pn['wording']
    for parent in place.get('includedIn', []):
        if parent.get('category', {}).get('id') == 'CON':
            record['county_ga'] = parent.get('nameGA')

    if record['name_ga'] and record['name_ga'] not in lookup_ga:
        lookup_ga[record['name_ga'].strip()] = record
        added += 1
    if record['genitive'] and record['genitive'] not in lookup_genitive:
        lookup_genitive[record['genitive'].strip()] = record
    if record['name_en'] and record['name_en'].lower() not in lookup_en:
        lookup_en[record['name_en'].strip().lower()] = record

print(f"New entries added: {added}")
print(f"Final lookup: {len(lookup_ga)} Irish, {len(lookup_en)} English entries")

with open(KG_FILES + 'logainm_lookup.pkl', 'wb') as f:
    pickle.dump({'ga': lookup_ga, 'en': lookup_en, 'genitive': lookup_genitive}, f)
print("Saved.")

Page 1: 1000 results (total so far: 1000/1898)
Page 2: 898 results (total so far: 1898/1898)

Fetched 1898 Wicklow places
New entries added: 0
Final lookup: 44262 Irish, 58880 English entries
Saved.


In [43]:
import pickle

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'

with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)

# Spot-check some known Wicklow placenames
test_en = ['wicklow', 'bray', 'greystones', 'arklow', 'enniskerry']
test_ga = ['Cill Mhantáin', 'Bré', 'Na Clocha Liatha']

print("English lookups:")
for name in test_en:
    hit = lookup['en'].get(name)
    print(f"  {name}: {'✓ ' + str(hit['county_ga']) if hit else '✗ missing'}")

print("\nIrish lookups:")
for name in test_ga:
    hit = lookup['ga'].get(name)
    print(f"  {name}: {'✓ ' + str(hit['county_ga']) if hit else '✗ missing'}")

English lookups:
  wicklow: ✓ Cill Mhantáin
  bray: ✓ Cill Mhantáin
  greystones: ✓ Cill Mhantáin
  arklow: ✓ Loch Garman
  enniskerry: ✓ Cill Mhantáin

Irish lookups:
  Cill Mhantáin: ✓ Cill Mhantáin
  Bré: ✓ Cill Mhantáin
  Na Clocha Liatha: ✓ Tiobraid Árann


In [46]:
import pandas as pd

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'

loc_nodes = pd.read_csv(KG_FILES + 'loc_nodes.csv')
print("columns:", loc_nodes.columns.tolist())
print(loc_nodes.head(10))

columns: ['qid', 'label_en', 'label_ga', 'description_en', 'instance_of_qids', 'parent_qid', 'country_qid', 'logainm_id', 'entity', 'canonical']
          qid                                           label_en  \
0     Q158490                                          Gaeltacht   
1       Q1761                                             Dublin   
2       Q1761                                             Dublin   
3  Q104334277                                        Carrowroger   
4       Q1761                                             Dublin   
5  Q100576994  Gaeltachta (Feidhmeanna Aire a Tharmligean), 1993   
6   Q54376793                                           Conamara   
7   Q54376793                                           Conamara   
8    Q1424587                                         Inis Meáin   
9     Q693164                                      Gaoth Dobhair   

            label_ga                                description_en  \
0      an Ghaeltacht   primarily Iri

In [51]:
import pandas as pd
import pickle

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'

# Load
with open(KG_FILES + 'logainm_lookup.pkl', 'rb') as f:
    lookup = pickle.load(f)
lookup_ga  = lookup['ga']
lookup_en  = lookup['en']
lookup_gen = lookup['genitive']

loc_nodes = pd.read_csv(KG_FILES + 'loc_nodes.csv')

# Deduplicate to one row per QID for matching
unique = loc_nodes.drop_duplicates(subset='qid')[['qid','label_en','label_ga','canonical','entity']].copy()

LENITION_MAP = {'bh': 'b', 'ch': 'c', 'dh': 'd', 'fh': 'f', 'gh': 'g',
                'mh': 'm', 'ph': 'p', 'sh': 's', 'th': 't'}

def delenite(word):
    for lenited, base in LENITION_MAP.items():
        if word.lower().startswith(lenited):
            return base + word[2:]
    return word

def match(row):
    # 1. label_ga / canonical exact
    for col in ['label_ga', 'canonical']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip() in lookup_ga:
            return lookup_ga[str(val).strip()]
    # 2. genitive
    for col in ['label_ga', 'entity']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip() in lookup_gen:
            return lookup_gen[str(val).strip()]
    # 3. Strip "Contae " prefix + de-lenition
    val = row.get('label_ga')
    if pd.notna(val):
        stripped = str(val).strip().removeprefix('Contae ').strip()
        delenited = delenite(stripped)
        for candidate in [stripped, delenited]:
            if candidate in lookup_ga:
                return lookup_ga[candidate]
            if candidate in lookup_gen:
                return lookup_gen[candidate]
    # 4. English fallback
    val = row.get('label_en')
    if pd.notna(val):
        en = str(val).strip().lower()
        if en in lookup_en:
            return lookup_en[en]
        stripped = en.removeprefix('county ').strip()
        if stripped in lookup_en:
            return lookup_en[stripped]
    return None

unique['_match'] = unique.apply(match, axis=1)

# Unpack match results
unique['logainm_id_new']  = unique['_match'].apply(lambda x: x['logainm_id']  if x else None)
unique['name_ga_lg']      = unique['_match'].apply(lambda x: x['name_ga']     if x else None)
unique['name_en_lg']      = unique['_match'].apply(lambda x: x['name_en']     if x else None)
unique['genitive_lg']     = unique['_match'].apply(lambda x: x['genitive']    if x else None)
unique['county_ga']       = unique['_match'].apply(lambda x: x['county_ga']   if x else None)
unique['gaeltacht']       = unique['_match'].apply(lambda x: x['gaeltacht']   if x else None)
unique['category_lg']     = unique['_match'].apply(lambda x: x['category']    if x else None)
unique.drop(columns='_match', inplace=True)

matched   = unique['logainm_id_new'].notna().sum()
unmatched = unique['logainm_id_new'].isna().sum()
print(f"Unique QIDs: {len(unique)}")
print(f"Matched:   {matched} ({matched/len(unique)*100:.1f}%)")
print(f"Unmatched: {unmatched} ({unmatched/len(unique)*100:.1f}%)")
print("\nSample matches:")
print(unique[unique['logainm_id_new'].notna()][['qid','label_en','label_ga','county_ga','gaeltacht','category_lg']].head(10))
print("\nSample unmatched:")
print(unique[unique['logainm_id_new'].isna()][['qid','label_en','label_ga']].head(10))

COUNTY_OVERRIDES = {
    'Q184469': {'logainm_id': 100013, 'name_ga': 'Ciarraí',       'name_en': 'Kerry',   'county_ga': 'Ciarraí',       'gaeltacht': False, 'genitive': 'Chiarraí',      'category': 'county'},
    'Q181862': {'logainm_id': 100005, 'name_ga': 'An Clár',        'name_en': 'Clare',   'county_ga': 'An Clár',        'gaeltacht': False, 'genitive': 'an Chláir',     'category': 'county'},
    'Q169923': {'logainm_id': 100012, 'name_ga': 'Gaillimh',       'name_en': 'Galway',  'county_ga': 'Gaillimh',       'gaeltacht': False, 'genitive': 'na Gaillimhe',  'category': 'county'},
    'Q179424': {'logainm_id': 100008, 'name_ga': 'Cill Mhantáin',  'name_en': 'Wicklow', 'county_ga': 'Cill Mhantáin',  'gaeltacht': False, 'genitive': 'Chill Mhantáin','category': 'county'},
}

# Apply overrides after the main match loop, before saving
for qid, override in COUNTY_OVERRIDES.items():
    if qid in unique['qid'].values:
        for col, val in [('logainm_id_new', override['logainm_id']),
                         ('name_ga_lg',     override['name_ga']),
                         ('name_en_lg',     override['name_en']),
                         ('genitive_lg',    override['genitive']),
                         ('county_ga',      override['county_ga']),
                         ('gaeltacht',      override['gaeltacht']),
                         ('category_lg',    override['category'])]:
            unique.loc[unique['qid'] == qid, col] = val

matched   = unique['logainm_id_new'].notna().sum()
unmatched = unique['logainm_id_new'].isna().sum()
print(f"After overrides — Matched: {matched} ({matched/len(unique)*100:.1f}%), Unmatched: {unmatched}")

# Save
unique.drop(columns=['label_en','label_ga','canonical','entity']).to_csv(
    KG_FILES + 'loc_logainm_enriched.csv', index=False
)
print(f"\nSaved → {KG_FILES}loc_logainm_enriched.csv")

Unique QIDs: 185
Matched:   122 (65.9%)
Unmatched: 63 (34.1%)

Sample matches:
           qid       label_en           label_ga          county_ga gaeltacht  \
1        Q1761         Dublin  Baile Átha Cliath  Baile Átha Cliath     False   
3   Q104334277    Carrowroger     Ceathrú Ruairí           Maigh Eo     False   
6    Q54376793       Conamara           Conamara           Gaillimh      True   
8     Q1424587     Inis Meáin         Inis Meáin           Maigh Eo     False   
9      Q693164  Gaoth Dobhair      Gaoth Dobhair       Dún na nGall      True   
10     Q129610         Galway           Gaillimh           Gaillimh     False   
11   Q59729282       Portumna          Port Omna           Gaillimh     False   
13     Q178626    County Mayo   Contae Mhaigh Eo           Maigh Eo     False   
14      Q36647           Cork           Corcaigh       Dún na nGall     False   
17     Q369911        Athlone   Baile Átha Luain         Ros Comáin     False   

           category_lg  
1   

In [50]:
# Check what's actually in the lookup for Kerry
print("Direct lookup_ga 'Ciarraí':", lookup_ga.get('Ciarraí'))
print("Direct lookup_ga 'Chiarraí':", lookup_ga.get('Chiarraí'))
print("Direct lookup_gen 'Chiarraí':", lookup_gen.get('Chiarraí'))
print("Direct lookup_en 'kerry':", lookup_en.get('kerry'))

# Check what delenite produces
print("\ndelenite('Chiarraí'):", delenite('Chiarraí'))

# Search for anything Kerry-like in the lookups
kerry_ga = [k for k in lookup_ga if 'iarraí' in k or 'iarrai' in k.lower()]
kerry_en = [k for k in lookup_en if 'kerry' in k.lower()]
print("\nlookup_ga keys containing 'iarraí':", kerry_ga[:10])
print("lookup_en keys containing 'kerry':", kerry_en[:10])

Direct lookup_ga 'Ciarraí': None
Direct lookup_ga 'Chiarraí': None
Direct lookup_gen 'Chiarraí': None
Direct lookup_en 'kerry': None

delenite('Chiarraí'): ciarraí

lookup_ga keys containing 'iarraí': ['Oileán Ciarraí', 'Gort Droma Ciarraí', 'Drom Ciarraí', 'Cill Chiarraí', 'Ceann Chiarraí', 'Oileán Ciarraí–Corca Dhuibhne', 'Ciarraí Cuirche', 'Paidhc na gCiarraíoch', 'Currach an Chiarraígh', 'Baile na gCiarraíoch']
lookup_en keys containing 'kerry': ['kerrymount', 'drumskerry', 'aghnaskerry', 'bennekerry', 'dromkerry', 'kilkerry', 'kerry way', 'kerry head', 'kerryhead', 'enniskerry']


In [52]:
enriched = pd.read_csv(KG_FILES + 'loc_logainm_enriched.csv')
print(enriched[enriched['logainm_id_new'].notna()].columns.tolist())
print(f"\nRows with logainm data: {enriched['logainm_id_new'].notna().sum()}")
print(f"Gaeltacht locations: {enriched['gaeltacht'].sum()}")
print(f"Categories:\n{enriched['category_lg'].value_counts()}")

['qid', 'logainm_id_new', 'name_ga_lg', 'name_en_lg', 'genitive_lg', 'county_ga', 'gaeltacht', 'category_lg']

Rows with logainm data: 123
Gaeltacht locations: 15
Categories:
category_lg
townland                     46
electoral division           18
population centre            12
town                          9
municipal district            5
island or archipelago         4
locality                      4
barony                        4
county                        4
monument                      3
civil parish                  3
borough district              2
city                          2
public park, sports field     1
metropolitan district         1
diocese                       1
bay                           1
lake or lakes                 1
promontory                    1
estuary                       1
Name: count, dtype: int64


In [53]:
import pandas as pd
from neo4j import GraphDatabase

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
URI      = "neo4j://127.0.0.1:7687"
AUTH     = ("neo4j", "hhdp6171")

enriched = pd.read_csv(KG_FILES + 'loc_logainm_enriched.csv')
matched  = enriched[enriched['logainm_id_new'].notna()].copy()

driver = GraphDatabase.driver(URI, auth=AUTH)

def update_loc_nodes(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (l:LOC {qid: row.qid})
        SET l.logainm_id  = row.logainm_id,
            l.name_ga_lg  = row.name_ga,
            l.name_en_lg  = row.name_en,
            l.genitive    = row.genitive,
            l.county_ga   = row.county_ga,
            l.gaeltacht   = row.gaeltacht,
            l.category_lg = row.category_lg
    """, rows=rows)

def add_located_in_edges(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (l:LOC {qid: row.qid})
        MATCH (c:LOC {name_ga_lg: row.county_ga})
        WHERE row.county_ga IS NOT NULL
        MERGE (l)-[:LOCATED_IN]->(c)
    """, rows=rows)

rows = [
    {
        'qid':         row.qid,
        'logainm_id':  int(row.logainm_id_new),
        'name_ga':     row.name_ga_lg   if pd.notna(row.name_ga_lg)  else None,
        'name_en':     row.name_en_lg   if pd.notna(row.name_en_lg)  else None,
        'genitive':    row.genitive_lg  if pd.notna(row.genitive_lg) else None,
        'county_ga':   row.county_ga    if pd.notna(row.county_ga)   else None,
        'gaeltacht':   bool(row.gaeltacht),
        'category_lg': row.category_lg  if pd.notna(row.category_lg) else None,
    }
    for row in matched.itertuples()
]

with driver.session(database="kg-phase3-logainm") as session:
    session.execute_write(update_loc_nodes, rows)
    print(f"Updated {len(rows)} LOC nodes with Logainm properties")

    session.execute_write(add_located_in_edges, rows)
    print("LOCATED_IN edges added")

driver.close()

# Verify
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="kg-phase3-logainm") as session:
    result = session.run("MATCH (l:LOC) WHERE l.logainm_id IS NOT NULL RETURN count(l) AS n")
    print(f"LOC nodes with logainm_id: {result.single()['n']}")

    result = session.run("MATCH ()-[:LOCATED_IN]->() RETURN count(*) AS n")
    print(f"LOCATED_IN edges: {result.single()['n']}")
driver.close()

Updated 123 LOC nodes with Logainm properties


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `LOC` does not exist in database `kg-phase3-logainm`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (l:LOC) WHERE l.logainm_id IS NOT NULL RETURN count(l) AS n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `logainm_id` does not exist in database `kg-phase3-logainm`. Verify that the spelling is correct.', positio

LOCATED_IN edges added
LOC nodes with logainm_id: 0
LOCATED_IN edges: 84


In [54]:
import pandas as pd
from neo4j import GraphDatabase

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
URI      = "neo4j://127.0.0.1:7687"
AUTH     = ("neo4j", "hhdp6171")

enriched = pd.read_csv(KG_FILES + 'loc_logainm_enriched.csv')
matched  = enriched[enriched['logainm_id_new'].notna()].copy()

driver = GraphDatabase.driver(URI, auth=AUTH)

def update_loc_nodes(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (l:Location {wikidata_id: row.qid})
        SET l.logainm_id  = row.logainm_id,
            l.name_ga_lg  = row.name_ga,
            l.name_en_lg  = row.name_en,
            l.genitive    = row.genitive,
            l.county_ga   = row.county_ga,
            l.gaeltacht   = row.gaeltacht,
            l.category_lg = row.category_lg
    """, rows=rows)

def add_located_in_edges(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (l:Location {wikidata_id: row.qid})
        MATCH (c:Location {name_ga_lg: row.county_ga})
        WHERE row.county_ga IS NOT NULL
        MERGE (l)-[:LOCATED_IN]->(c)
    """, rows=rows)

rows = [
    {
        'qid':         row.qid,
        'logainm_id':  int(row.logainm_id_new),
        'name_ga':     row.name_ga_lg   if pd.notna(row.name_ga_lg)  else None,
        'name_en':     row.name_en_lg   if pd.notna(row.name_en_lg)  else None,
        'genitive':    row.genitive_lg  if pd.notna(row.genitive_lg) else None,
        'county_ga':   row.county_ga    if pd.notna(row.county_ga)   else None,
        'gaeltacht':   bool(row.gaeltacht),
        'category_lg': row.category_lg  if pd.notna(row.category_lg) else None,
    }
    for row in matched.itertuples()
]

with driver.session(database="neo4j") as session:
    session.execute_write(update_loc_nodes, rows)
    print(f"Updated {len(rows)} Location nodes with Logainm properties")

    session.execute_write(add_located_in_edges, rows)
    print("LOCATED_IN edges added")

driver.close()

# Verify
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    result = session.run("MATCH (l:Location) WHERE l.logainm_id IS NOT NULL RETURN count(l) AS n")
    print(f"Location nodes with logainm_id: {result.single()['n']}")

    result = session.run("MATCH ()-[:LOCATED_IN]->() RETURN count(*) AS n")
    print(f"LOCATED_IN edges: {result.single()['n']}")
driver.close()

Updated 123 Location nodes with Logainm properties
LOCATED_IN edges added
Location nodes with logainm_id: 123
LOCATED_IN edges: 162


In [55]:
import pandas as pd

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'

crossref = pd.read_csv(KG_FILES + 'per_oireachtas_crossref.csv')
oireachtas_qids = pd.read_csv(KG_FILES + 'oireachtas_qids.csv')

print("per_oireachtas_crossref columns:", crossref.columns.tolist())
print(crossref.head(5))
print(f"\nRows: {len(crossref)}")

print("\noireachtas_qids columns:", oireachtas_qids.columns.tolist())
print(oireachtas_qids.head(5))
print(f"\nRows: {len(oireachtas_qids)}")

per_oireachtas_crossref columns: ['entity', 'match_type', 'matched_name', 'score', 'party', 'constituency', 'train_count']
       entity    match_type           matched_name  score        party  \
0  Aire Stáit    ROLE_TITLE                    NaN    NaN          NaN   
1   Taoiseach    ROLE_TITLE                    NaN    NaN          NaN   
2        Aire    ROLE_TITLE                    NaN    NaN          NaN   
3       tAire    ROLE_TITLE                    NaN    NaN          NaN   
4   Humphreys  SURNAME_ONLY  Dr. Francis Humphreys  100.0  Fianna Fáil   

      constituency  train_count  
0              NaN           10  
1              NaN            6  
2              NaN            5  
3              NaN            5  
4  Carlow-Kilkenny            5  

Rows: 651

oireachtas_qids columns: ['member_code', 'full_name_en', 'party', 'constituency', 'qid', 'wikidata_description', 'confident']
                     member_code        full_name_en        party  \
0  Henry-J-J-Abbott.D

In [56]:
import requests

member_code = "Gerry-Adams.D.2011-03-09"

r = requests.get(
    f"https://api.oireachtas.ie/v1/members",
    params={"member_id": member_code, "limit": 1},
    timeout=10
)
print(r.status_code)
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:2000])

200
{
  "head": {
    "counts": {
      "memberCount": 0,
      "resultCount": 0
    }
  },
  "results": []
}


In [57]:
import requests, json

# Try the correct member endpoint with member_code
member_code = "GerryAdams"

# Try a few different endpoint patterns
endpoints = [
    f"https://api.oireachtas.ie/v1/members?member_id={member_code}",
    f"https://api.oireachtas.ie/v1/members?memberCode={member_code}",
    f"https://api.oireachtas.ie/v1/members?name=Gerry Adams&limit=3",
    f"https://api.oireachtas.ie/v1/members?limit=1",  # just to see structure
]

for url in endpoints:
    r = requests.get(url, timeout=10)
    data = r.json()
    count = data.get('head', {}).get('counts', {})
    results = data.get('results', [])
    print(f"\nURL: {url}")
    print(f"  counts: {count}")
    if results:
        print(json.dumps(results[0], indent=2, ensure_ascii=False)[:500])


URL: https://api.oireachtas.ie/v1/members?member_id=GerryAdams
  counts: {'memberCount': 0, 'resultCount': 0}

URL: https://api.oireachtas.ie/v1/members?memberCode=GerryAdams
  counts: {'memberCount': 1928, 'resultCount': 1928}
{
  "member": {
    "gender": "",
    "uri": "https://data.oireachtas.ie/ie/oireachtas/member/id/Henry-J-J-Abbott.D.1987-03-10",
    "pId": "HenryJJAbbott",
    "firstName": "Henry J. J.",
    "lastName": "Abbott",
    "image": false,
    "wikiTitle": null,
    "memberships": [
      {
        "membership": {
          "house": {
            "uri": "https://data.oireachtas.ie/ie/oireachtas/house/dail/25",
            "houseCode": "dail",
            "houseNo": "25",
            "chamberType": "h

URL: https://api.oireachtas.ie/v1/members?name=Gerry Adams&limit=3
  counts: {'memberCount': 1928, 'resultCount': 1928}
{
  "member": {
    "gender": "",
    "uri": "https://data.oireachtas.ie/ie/oireachtas/member/id/Henry-J-J-Abbott.D.1987-03-10",
    "pId": "HenryJJA

In [58]:
import pandas as pd
from neo4j import GraphDatabase

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

crossref = pd.read_csv(KG_FILES + 'per_oireachtas_crossref.csv')
oireachtas = pd.read_csv(KG_FILES + 'oireachtas_qids.csv')

# Only confident matches with QIDs
confident = oireachtas[
    (oireachtas['confident'] == True) &
    (oireachtas['qid'].notna()) &
    (oireachtas['party'].notna())
].copy()

print(f"Confident PER matches: {len(confident)}")
print(f"Unique parties: {confident['party'].nunique()}")
print(f"Unique constituencies: {confident['constituency'].nunique()}")
print(confident[['full_name_en','party','constituency','qid']].head(10))

Confident PER matches: 1295
Unique parties: 26
Unique constituencies: 121
        full_name_en         party          constituency         qid
1   Caroline Acheson   Fianna Fáil       Tipperary South    Q5046195
2        Gerry Adams     Sinn Féin                 Louth      Q76139
4      Garret Ahearn     Fine Gael  Administrative Panel   Q89368401
5     Theresa Ahearn     Fine Gael       Tipperary South    Q7782845
6       Bertie Ahern   Fianna Fáil        Dublin Central     Q154550
7       Ciarán Ahern  Labour Party     Dublin South-West  Q106418148
8       Dermot Ahern   Fianna Fáil                 Louth     Q974143
10        Liam Ahern   Fianna Fáil       Cork North East    Q6539473
11     Michael Ahern   Fianna Fáil             Cork East    Q1672642
12        Noel Ahern   Fianna Fáil     Dublin North-West    Q1410728


In [59]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

qids = confident['qid'].tolist()

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    result = session.run("""
        UNWIND $qids AS qid
        MATCH (p:Person {wikidata_id: qid})
        RETURN count(p) AS matched
    """, qids=qids)
    print(f"Person nodes in graph matching confident QIDs: {result.single()['matched']}")

    # Also check what Person nodes look like
    result2 = session.run("MATCH (p:Person) RETURN properties(p) LIMIT 3")
    for r in result2:
        print(r['properties(p)'])
driver.close()

Person nodes in graph matching confident QIDs: 1295
{'party_qid': 'Q216517', 'party_name': 'Fianna Fáil', 'description_en': 'Irish politician', 'full_name_en': 'Caroline Acheson', 'wikidata_id': 'Q5046195', 'label_en': 'Carrie Acheson', 'label_ga': 'Carrie Acheson', 'constituency_name': 'Tipperary South', 'node_id': 'Q5046195', 'is_stub': False}
{'party_qid': 'Q76382', 'party_name': 'Sinn Féin', 'description_en': 'Irish republican politician (born 1948)', 'full_name_en': 'Gerry Adams', 'wikidata_id': 'Q76139', 'label_en': 'Gerry Adams', 'label_ga': 'Gearóid Mac Ádhaimh', 'constituency_name': 'Louth', 'node_id': 'Q76139', 'is_stub': False}
{'party_name': 'Fine Gael', 'description_en': 'Irish politician', 'full_name_en': 'Garret Ahearn', 'wikidata_id': 'Q89368401', 'label_en': 'Garret Ahearn', 'constituency_name': 'Administrative Panel', 'node_id': 'Q89368401', 'is_stub': False}


In [61]:
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    # Check if party organisations exist
    result = session.run("""
        MATCH (p:Person) WHERE p.party_name IS NOT NULL
        WITH DISTINCT p.party_name AS party, p.party_qid AS qid
        OPTIONAL MATCH (o:Organisation {wikidata_id: qid})
        RETURN party, qid, o.label_en AS org_found
        LIMIT 10
    """)
    print("Party → Organisation matches:")
    for r in result:
        print(f"  {r['party']} ({r['qid']}) → {r['org_found']}")

    # Check constituency → Location matches
    result2 = session.run("""
        MATCH (p:Person) WHERE p.constituency_name IS NOT NULL
        WITH DISTINCT p.constituency_name AS con
        OPTIONAL MATCH (l:Location {label_en: con})
        RETURN con, l.label_en AS loc_found
        LIMIT 10
    """)
    print("\nConstituency → Location matches:")
    for r in result2:
        print(f"  {r['con']} → {r['loc_found']}")
driver.close()

Party → Organisation matches:
  Fianna Fáil (Q216517) → Fianna Fáil
  Sinn Féin (Q76382) → Sinn Féin
  Fine Gael (None) → None
  Labour Party (Q503614) → None
  Cumann na nGaedheal (None) → None
  Fine Gael (Q247135) → Fine Gael
  Independent (None) → None
  Fianna Fáil (None) → None
  Sinn Féin (None) → None
  Fianna Fáil (Q488523) → None

Constituency → Location matches:
  Tipperary South → None
  Louth → None
  Administrative Panel → None
  Dublin Central → None
  Dublin South-West → None
  Cork North East → None
  Cork East → None
  Dublin North-West → None
  Laois → None
  Leix-Offaly → None


In [62]:
driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    # How many distinct parties and constituencies are we dealing with?
    result = session.run("""
        MATCH (p:Person)
        WHERE p.party_name IS NOT NULL
        RETURN count(DISTINCT p.party_name) AS parties,
               count(DISTINCT p.constituency_name) AS constituencies
    """)
    r = result.single()
    print(f"Distinct parties: {r['parties']}")
    print(f"Distinct constituencies: {r['constituencies']}")

    # Sample constituency names
    result2 = session.run("""
        MATCH (p:Person) WHERE p.constituency_name IS NOT NULL
        RETURN DISTINCT p.constituency_name AS con
        ORDER BY con LIMIT 20
    """)
    print("\nSample constituencies:")
    for r in result2:
        print(f"  {r['con']}")
driver.close()

Distinct parties: 25
Distinct constituencies: 121

Sample constituencies:
  1925
  1928
  1931
  1934
  Administrative Panel
  Agricultural Panel
  Athlone-Longford
  Carlow-Kilkenny
  Cavan
  Cavan-Monaghan
  Clare
  Clare-Galway South
  Cork Borough
  Cork City
  Cork East
  Cork Mid
  Cork Mid, North, South, South East and West
  Cork North
  Cork North East
  Cork North-Central


In [63]:
import pandas as pd
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)

def create_party_nodes_and_edges(tx):
    tx.run("""
        MATCH (p:Person) WHERE p.party_name IS NOT NULL
        MERGE (party:Organisation {label_en: p.party_name})
        ON CREATE SET party.node_id = 'PARTY_' + p.party_name,
                      party.wikidata_id = p.party_qid,
                      party.is_stub = true,
                      party.category = 'political_party'
        MERGE (p)-[:MEMBER_OF_PARTY]->(party)
    """)

def create_constituency_nodes_and_edges(tx):
    tx.run("""
        MATCH (p:Person) WHERE p.constituency_name IS NOT NULL
        MERGE (con:Constituency {name: p.constituency_name})
        ON CREATE SET con.node_id = 'CON_' + p.constituency_name,
                      con.is_stub = true
        MERGE (p)-[:REPRESENTS]->(con)
    """)

with driver.session(database="neo4j") as session:
    session.execute_write(create_party_nodes_and_edges)
    print("MEMBER_OF_PARTY edges created")

    session.execute_write(create_constituency_nodes_and_edges)
    print("REPRESENTS edges created")

# Verify
with driver.session(database="neo4j") as session:
    r = session.run("MATCH ()-[:MEMBER_OF_PARTY]->() RETURN count(*) AS n").single()
    print(f"MEMBER_OF_PARTY edges: {r['n']}")

    r = session.run("MATCH ()-[:REPRESENTS]->() RETURN count(*) AS n").single()
    print(f"REPRESENTS edges: {r['n']}")

    r = session.run("MATCH (c:Constituency) RETURN count(c) AS n").single()
    print(f"Constituency nodes created: {r['n']}")

    r = session.run("MATCH (o:Organisation) RETURN count(o) AS n").single()
    print(f"Organisation nodes total: {r['n']}")

driver.close()

MEMBER_OF_PARTY edges created
REPRESENTS edges created
MEMBER_OF_PARTY edges: 1272
REPRESENTS edges: 1291
Constituency nodes created: 121
Organisation nodes total: 131


In [64]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:

    # Node counts by label
    print("=== NODE COUNTS ===")
    for label in ["Person", "Location", "Organisation", "Constituency", "StubNode", "Office"]:
        r = session.run(f"MATCH (n:{label}) RETURN count(n) AS n").single()
        print(f"  {label}: {r['n']}")

    # Edge counts by type
    print("\n=== EDGE COUNTS ===")
    r = session.run("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType")
    rel_types = [row['relationshipType'] for row in r]
    for rel in rel_types:
        r = session.run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n").single()
        print(f"  {rel}: {r['n']}")

    # New Logainm properties
    print("\n=== LOGAINM ENRICHMENT ===")
    r = session.run("MATCH (l:Location) WHERE l.logainm_id IS NOT NULL RETURN count(l) AS n").single()
    print(f"  Location nodes with Logainm ID: {r['n']}")
    r = session.run("MATCH (l:Location) WHERE l.gaeltacht = true RETURN count(l) AS n").single()
    print(f"  Gaeltacht locations: {r['n']}")
    r = session.run("MATCH (l:Location) WHERE l.genitive IS NOT NULL RETURN count(l) AS n").single()
    print(f"  Locations with genitive form: {r['n']}")

    # Graph density indicators
    print("\n=== GRAPH SUMMARY ===")
    r = session.run("MATCH (n) RETURN count(n) AS n").single()
    total_nodes = r['n']
    r = session.run("MATCH ()-[r]->() RETURN count(r) AS n").single()
    total_edges = r['n']
    print(f"  Total nodes: {total_nodes}")
    print(f"  Total edges: {total_edges}")
    print(f"  Edges per node: {total_edges/total_nodes:.2f}")

    # Connectivity — nodes with at least one edge
    r = session.run("""
        MATCH (n) WHERE (n)--()
        RETURN count(DISTINCT n) AS n
    """).single()
    print(f"  Connected nodes: {r['n']} ({r['n']/total_nodes*100:.1f}%)")

driver.close()

=== NODE COUNTS ===
  Person: 1272
  Location: 185
  Organisation: 131
  Constituency: 121
  StubNode: 1478
  Office: 266

=== EDGE COUNTS ===
  MEMBER_OF: 588
  HOLDS_POSITION: 2551
  LOCATED_IN: 162
  HQ_IN: 32
  REPRESENTS: 1291
  MEMBER_OF_PARTY: 1272

=== LOGAINM ENRICHMENT ===
  Location nodes with Logainm ID: 123
  Gaeltacht locations: 15
  Locations with genitive form: 123

=== GRAPH SUMMARY ===
  Total nodes: 3453
  Total edges: 5896
  Edges per node: 1.71
  Connected nodes: 1828 (52.9%)


In [66]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    r = session.run("""
        MATCH (n:StubNode)
        OPTIONAL MATCH (n)-[r]-()
        RETURN count(DISTINCT n) AS stubs,
               count(r) AS stub_edges,
               count(DISTINCT CASE WHEN r IS NOT NULL THEN n END) AS connected_stubs
    """).single()
    print(f"StubNodes: {r['stubs']}")
    print(f"StubNode edges: {r['stub_edges']}")
    print(f"Connected StubNodes: {r['connected_stubs']}")
driver.close()

StubNodes: 1478
StubNode edges: 0
Connected StubNodes: 0


In [67]:
import requests

def fetch_wikidata_relations(qid):
    query = f"""
    SELECT ?borninID ?borninLabel ?educatedatID ?educatedatLabel ?succeededbyID ?succeededbyLabel WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P19 ?bornin. BIND(REPLACE(STR(?bornin), ".*Q", "Q") AS ?borninID) }}
      OPTIONAL {{ wd:{qid} wdt:P69 ?educatedat. BIND(REPLACE(STR(?educatedat), ".*Q", "Q") AS ?educatedatID) }}
      OPTIONAL {{ wd:{qid} wdt:P1366 ?succeededby. BIND(REPLACE(STR(?succeededby), ".*Q", "Q") AS ?succeededbyID) }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,ga". }}
    }}
    """
    r = requests.get(
        "https://query.wikidata.org/sparql",
        params={"query": query, "format": "json"},
        headers={"User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"},
        timeout=30
    )
    return r.json().get('results', {}).get('bindings', [])

# Test with Bertie Ahern
results = fetch_wikidata_relations("Q154550")
for r in results:
    print(r)

{'educatedatID': {'type': 'literal', 'value': 'Q7586723'}, 'borninID': {'type': 'literal', 'value': 'Q1261255'}, 'borninLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Drumcondra'}, 'educatedatLabel': {'xml:lang': 'en', 'type': 'literal', 'value': "St. Aidan's C.B.S."}}
{'educatedatID': {'type': 'literal', 'value': 'Q3098071'}, 'borninID': {'type': 'literal', 'value': 'Q1261255'}, 'borninLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Drumcondra'}, 'educatedatLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Dublin Institute of Technology'}}


In [68]:
import pandas as pd
import requests
import time
from neo4j import GraphDatabase

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/KG/'
URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

oireachtas = pd.read_csv(KG_FILES + 'oireachtas_qids.csv')
confident = oireachtas[
    (oireachtas['confident'] == True) &
    (oireachtas['qid'].notna())
].drop_duplicates(subset='qid').copy()

print(f"Fetching Wikidata relations for {len(confident)} persons...")

def fetch_wikidata_relations(qid):
    query = f"""
    SELECT ?borninID ?borninLabel ?educatedatID ?educatedatLabel ?succeededbyID ?succeededbyLabel WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P19 ?bornin. BIND(REPLACE(STR(?bornin), ".*Q", "Q") AS ?borninID) }}
      OPTIONAL {{ wd:{qid} wdt:P69 ?educatedat. BIND(REPLACE(STR(?educatedat), ".*Q", "Q") AS ?educatedatID) }}
      OPTIONAL {{ wd:{qid} wdt:P1366 ?succeededby. BIND(REPLACE(STR(?succeededby), ".*Q", "Q") AS ?succeededbyID) }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,ga". }}
    }}
    """
    r = requests.get(
        "https://query.wikidata.org/sparql",
        params={"query": query, "format": "json"},
        headers={"User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"},
        timeout=30
    )
    return r.json().get('results', {}).get('bindings', [])

# Fetch all relations
all_relations = []
for i, row in confident.iterrows():
    qid = row['qid']
    try:
        bindings = fetch_wikidata_relations(qid)
        for b in bindings:
            rel = {'person_qid': qid}
            if 'borninID' in b:
                rel['born_in_qid']    = b['borninID']['value']
                rel['born_in_label']  = b.get('borninLabel', {}).get('value')
            if 'educatedatID' in b:
                rel['educated_at_qid']   = b['educatedatID']['value']
                rel['educated_at_label'] = b.get('educatedatLabel', {}).get('value')
            if 'succeededbyID' in b:
                rel['succeeded_by_qid']   = b['succeededbyID']['value']
                rel['succeeded_by_label'] = b.get('succeededbyLabel', {}).get('value')
            all_relations.append(rel)
        if i % 100 == 0:
            print(f"  {i}/{len(confident)} done...")
        time.sleep(0.3)
    except Exception as e:
        print(f"  Error {qid}: {e}")
        time.sleep(2)

df = pd.DataFrame(all_relations)
print(f"\nTotal relation rows: {len(df)}")
print(f"BORN_IN:      {df['born_in_qid'].notna().sum() if 'born_in_qid' in df else 0}")
print(f"EDUCATED_AT:  {df['educated_at_qid'].notna().sum() if 'educated_at_qid' in df else 0}")
print(f"SUCCEEDED_BY: {df['succeeded_by_qid'].notna().sum() if 'succeeded_by_qid' in df else 0}")

df.to_csv(KG_FILES + 'per_wikidata_relations.csv', index=False)
print(f"Saved → {KG_FILES}per_wikidata_relations.csv")

# Now push to Neo4j
driver = GraphDatabase.driver(URI, auth=AUTH)

def add_born_in(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (p:Person {wikidata_id: row.person_qid})
        MERGE (l:Location {wikidata_id: row.born_in_qid})
        ON CREATE SET l.label_en = row.born_in_label,
                      l.node_id  = row.born_in_qid,
                      l.is_stub  = true
        MERGE (p)-[:BORN_IN]->(l)
    """, rows=rows)

def add_educated_at(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (p:Person {wikidata_id: row.person_qid})
        MERGE (o:Organisation {wikidata_id: row.educated_at_qid})
        ON CREATE SET o.label_en = row.educated_at_label,
                      o.node_id  = row.educated_at_qid,
                      o.is_stub  = true
        MERGE (p)-[:EDUCATED_AT]->(o)
    """, rows=rows)

def add_succeeded_by(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MATCH (p:Person {wikidata_id: row.person_qid})
        MERGE (s:Person {wikidata_id: row.succeeded_by_qid})
        ON CREATE SET s.label_en = row.succeeded_by_label,
                      s.node_id  = row.succeeded_by_qid,
                      s.is_stub  = true
        MERGE (p)-[:SUCCEEDED_BY]->(s)
    """, rows=rows)

with driver.session(database="neo4j") as session:
    born_rows = [
        {'person_qid': r.person_qid, 'born_in_qid': r.born_in_qid, 'born_in_label': r.born_in_label}
        for r in df[df.get('born_in_qid', pd.Series()).notna()].itertuples()
        if hasattr(r, 'born_in_qid') and pd.notna(r.born_in_qid)
    ] if 'born_in_qid' in df else []

    edu_rows = [
        {'person_qid': r.person_qid, 'educated_at_qid': r.educated_at_qid, 'educated_at_label': r.educated_at_label}
        for r in df[df.get('educated_at_qid', pd.Series()).notna()].itertuples()
        if hasattr(r, 'educated_at_qid') and pd.notna(r.educated_at_qid)
    ] if 'educated_at_qid' in df else []

    suc_rows = [
        {'person_qid': r.person_qid, 'succeeded_by_qid': r.succeeded_by_qid, 'succeeded_by_label': r.succeeded_by_label}
        for r in df[df.get('succeeded_by_qid', pd.Series()).notna()].itertuples()
        if hasattr(r, 'succeeded_by_qid') and pd.notna(r.succeeded_by_qid)
    ] if 'succeeded_by_qid' in df else []

    if born_rows:
        session.execute_write(add_born_in, born_rows)
        print(f"BORN_IN edges added: {len(born_rows)}")
    if edu_rows:
        session.execute_write(add_educated_at, edu_rows)
        print(f"EDUCATED_AT edges added: {len(edu_rows)}")
    if suc_rows:
        session.execute_write(add_succeeded_by, suc_rows)
        print(f"SUCCEEDED_BY edges added: {len(suc_rows)}")

driver.close()
print("Done.")

Fetching Wikidata relations for 1267 persons...
  100/1267 done...
  300/1267 done...
  Error Q2034158: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=30)
  Error Q5325552: Expecting value: line 1 column 1 (char 0)
  400/1267 done...
  Error Q5342420: Expecting value: line 1 column 1 (char 0)
  Error Q6829535: Expecting value: line 1 column 1 (char 0)
  Error Q6227579: Expecting value: line 1 column 1 (char 0)
  Error Q4960850: Expecting value: line 1 column 1 (char 0)
  Error Q16026989: Expecting value: line 1 column 1 (char 0)
  500/1267 done...
  600/1267 done...
  700/1267 done...
  Error Q4696759: Expecting value: line 1 column 1 (char 0)
  Error Q6831055: Expecting value: line 1 column 1 (char 0)
  900/1267 done...
  Error Q325381: Expecting value: line 1 column 1 (char 0)
  Error Q6396541: Expecting value: line 1 column 1 (char 0)
  Error Q540369: Expecting value: line 1 column 1 (char 0)
  1000/1267 done...
  1100/1267 done...
  1300/126

In [69]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    print("=== FINAL GRAPH STATS ===")
    
    r = session.run("MATCH (n) RETURN count(n) AS n").single()
    total_nodes = r['n']
    r = session.run("MATCH ()-[r]->() RETURN count(r) AS n").single()
    total_edges = r['n']
    print(f"Total nodes: {total_nodes}")
    print(f"Total edges: {total_edges}")
    print(f"Edges per node: {total_edges/total_nodes:.2f}")

    print("\n--- Edges by type ---")
    result = session.run("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType")
    for row in result:
        rel = row['relationshipType']
        r = session.run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n").single()
        print(f"  {rel}: {r['n']}")

    r = session.run("MATCH (n) WHERE (n)--() RETURN count(DISTINCT n) AS n").single()
    print(f"\nConnected nodes: {r['n']} ({r['n']/total_nodes*100:.1f}%)")
driver.close()

=== FINAL GRAPH STATS ===
Total nodes: 3966
Total edges: 7492
Edges per node: 1.89

--- Edges by type ---
  MEMBER_OF: 588
  HOLDS_POSITION: 2551
  LOCATED_IN: 162
  HQ_IN: 32
  REPRESENTS: 1291
  MEMBER_OF_PARTY: 1272
  BORN_IN: 871
  EDUCATED_AT: 725

Connected nodes: 2351 (59.3%)


In [71]:
import requests, json

r = requests.get(
    "https://api.oireachtas.ie/v1/committees",
    params={"limit": 5},
    timeout=10
)
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:3000])

ReadTimeout: HTTPSConnectionPool(host='api.oireachtas.ie', port=443): Read timed out. (read timeout=10)

In [72]:
import requests, json

# Try with longer timeout and different endpoints
for url in [
    "https://api.oireachtas.ie/v1/committees?limit=3",
    "https://api.oireachtas.ie/v1/memberships?limit=3",
]:
    try:
        r = requests.get(url, timeout=30)
        print(f"\n{url} → {r.status_code}")
        print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:1000])
    except Exception as e:
        print(f"\n{url} → Error: {e}")


https://api.oireachtas.ie/v1/committees?limit=3 → 404
{
  "message": "Not found."
}

https://api.oireachtas.ie/v1/memberships?limit=3 → 404
{
  "message": "Not found."
}


In [73]:
import requests, json

for endpoint in [
    "https://api.oireachtas.ie/v1/",
    "https://api.oireachtas.ie/v1/members?limit=1",
    "https://api.oireachtas.ie/v1/debates?limit=1",
    "https://api.oireachtas.ie/v1/legislation?limit=1",
    "https://api.oireachtas.ie/v1/committee_reports?limit=1",
]:
    try:
        r = requests.get(endpoint, timeout=30)
        print(f"{endpoint} → {r.status_code}")
        if r.status_code == 200:
            keys = list(r.json().keys())
            print(f"  keys: {keys}")
    except Exception as e:
        print(f"{endpoint} → Error: {e}")

https://api.oireachtas.ie/v1/ → 404
https://api.oireachtas.ie/v1/members?limit=1 → 200
  keys: ['head', 'results']
https://api.oireachtas.ie/v1/debates?limit=1 → 200
  keys: ['head', 'results']
https://api.oireachtas.ie/v1/legislation?limit=1 → 200
  keys: ['head', 'results']
https://api.oireachtas.ie/v1/committee_reports?limit=1 → 404


In [74]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    # Check actual properties on each node type
    for label in ["Person", "Location", "Organisation", "Constituency", "StubNode"]:
        result = session.run(f"MATCH (n:{label}) RETURN properties(n) LIMIT 1")
        row = result.single()
        if row:
            print(f"\n{label} properties:")
            print(list(row['properties(n)'].keys()))

driver.close()


Person properties:
['party_qid', 'party_name', 'description_en', 'full_name_en', 'wikidata_id', 'label_en', 'label_ga', 'constituency_name', 'node_id', 'is_stub']

Location properties:
['description_en', 'wikidata_id', 'label_en', 'label_ga', 'entity', 'node_id', 'is_stub']

Organisation properties:
['description_en', 'wikidata_id', 'hq_qid', 'label_en', 'label_ga', 'entity', 'node_id', 'is_stub']

Constituency properties:
['name', 'node_id', 'is_stub']

StubNode properties:
['node_type', 'stub_class', 'entity', 'node_id', 'is_stub']


In [75]:
from neo4j import GraphDatabase

URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)
with driver.session(database="neo4j") as session:
    result = session.run("""
        MATCH (n:Location) 
        WHERE n.logainm_id IS NOT NULL 
        RETURN properties(n) LIMIT 1
    """)
    row = result.single()
    if row:
        print("Enriched Location properties:")
        print(list(row['properties(n)'].keys()))
driver.close()

Enriched Location properties:
['logainm_id', 'description_en', 'wikidata_id', 'gaeltacht', 'name_ga_lg', 'label_en', 'name_en_lg', 'county_ga', 'category_lg', 'parent_qid', 'genitive', 'label_ga', 'entity', 'node_id', 'is_stub']


In [76]:
import pandas as pd
from neo4j import GraphDatabase

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/'
URI  = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "hhdp6171")

driver = GraphDatabase.driver(URI, auth=AUTH)

with driver.session(database="neo4j") as session:

    # Triples
    result = session.run("""
        MATCH (h)-[r]->(t)
        WHERE h.node_id IS NOT NULL AND t.node_id IS NOT NULL
        RETURN h.node_id AS head, type(r) AS relation, t.node_id AS tail
    """)
    triples = [(r['head'], r['relation'], r['tail']) for r in result]

    # loc_nodes
    result = session.run("""
        MATCH (n:Location)
        RETURN n.node_id AS qid, n.wikidata_id AS wikidata_id,
               n.label_en AS label_en, n.label_ga AS label_ga,
               n.entity AS entity, n.is_stub AS is_stub,
               n.parent_qid AS parent_qid, n.description_en AS description_en,
               n.logainm_id AS logainm_id, n.name_ga_lg AS name_ga_lg,
               n.name_en_lg AS name_en_lg, n.genitive AS genitive,
               n.county_ga AS county_ga, n.gaeltacht AS gaeltacht,
               n.category_lg AS category_lg
    """)
    loc_nodes = [dict(r) for r in result]

    # per_nodes
    result = session.run("""
        MATCH (n:Person)
        RETURN n.node_id AS qid, n.wikidata_id AS wikidata_id,
               n.label_en AS label_en, n.label_ga AS label_ga,
               n.full_name_en AS full_name_en, n.description_en AS description_en,
               n.party_name AS party_name, n.party_qid AS party_qid,
               n.constituency_name AS constituency_name, n.is_stub AS is_stub
    """)
    per_nodes = [dict(r) for r in result]

    # org_nodes
    result = session.run("""
        MATCH (n:Organisation)
        RETURN n.node_id AS qid, n.wikidata_id AS wikidata_id,
               n.label_en AS label_en, n.label_ga AS label_ga,
               n.entity AS entity, n.description_en AS description_en,
               n.hq_qid AS hq_qid, n.is_stub AS is_stub
    """)
    org_nodes = [dict(r) for r in result]

    # stub_nodes
    result = session.run("""
        MATCH (n:StubNode)
        RETURN n.node_id AS qid, n.entity AS entity,
               n.node_type AS node_type, n.stub_class AS stub_class,
               n.is_stub AS is_stub
    """)
    stub_nodes = [dict(r) for r in result]

driver.close()

# Save triples
df_triples = pd.DataFrame(triples, columns=['head', 'relation', 'tail'])
df_triples.to_csv(KG_FILES + 'kg_triples.tsv', sep='\t', index=False, header=False)
print(f"kg_triples.tsv: {len(df_triples)} triples")

# Save node CSVs
pd.DataFrame(loc_nodes).to_csv(KG_FILES + 'loc_nodes.csv', index=False)
print(f"loc_nodes.csv: {len(loc_nodes)} rows")

pd.DataFrame(per_nodes).to_csv(KG_FILES + 'per_nodes.csv', index=False)
print(f"per_nodes.csv: {len(per_nodes)} rows")

pd.DataFrame(org_nodes).to_csv(KG_FILES + 'org_nodes.csv', index=False)
print(f"org_nodes.csv: {len(org_nodes)} rows")

pd.DataFrame(stub_nodes).to_csv(KG_FILES + 'stub_nodes.csv', index=False)
print(f"stub_nodes.csv: {len(stub_nodes)} rows")

kg_triples.tsv: 7492 triples
loc_nodes.csv: 487 rows
per_nodes.csv: 1272 rows
org_nodes.csv: 342 rows
stub_nodes.csv: 1478 rows


In [78]:
import pickle
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/'

# Load triples
tf = TriplesFactory.from_path(KG_FILES + 'kg_triples.tsv')

# Split into train/val/test
training, testing, validation = tf.split([0.8, 0.1, 0.1], random_state=42)

print(f"Training triples: {training.num_triples}")
print(f"Validation triples: {validation.num_triples}")
print(f"Test triples: {testing.num_triples}")
print(f"Entities: {tf.num_entities}")
print(f"Relations: {tf.num_relations}")

# Train TransE — same settings as Phase 2
result = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='TransE',
    model_kwargs=dict(embedding_dim=128),
    optimizer='Adam',
    optimizer_kwargs=dict(lr=0.001),
    training_kwargs=dict(num_epochs=100, batch_size=256),
    stopper='early',
    stopper_kwargs=dict(frequency=10, patience=3, relative_delta=0.001),
    random_seed=42,
    device='cpu',
)

print(f"\nTraining complete.")
print(f"Hits@10: {result.get_metric('hits@10'):.4f}")
print(f"MRR: {result.get_metric('mean_reciprocal_rank'):.4f}")

# Extract embeddings keyed by node_id
entity_embeddings = result.model.entity_representations[0]().detach().cpu().numpy()
entity_to_id = result.training.entity_to_id

id_to_entity = {v: k for k, v in entity_to_id.items()}
qid_to_embedding = {
    id_to_entity[i]: entity_embeddings[i]
    for i in range(len(entity_embeddings))
}

print(f"Embedding dict: {len(qid_to_embedding)} entities × {entity_embeddings.shape[1]} dims")

# Save
with open(KG_FILES + 'TransE_qid_embeddings.pkl', 'wb') as f:
    pickle.dump(qid_to_embedding, f)
print(f"Saved → {KG_FILES}TransE_qid_embeddings.pkl")

Training triples: 5993
Validation triples: 750
Test triples: 749
Entities: 2351
Relations: 8


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/24.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/24.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/24.0 [00:00<?, ?batch/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.41s seconds


Evaluating on cpu:   0%|          | 0.00/749 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.41s seconds
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Training complete.
Hits@10: 0.2477
MRR: 0.0895
Embedding dict: 2351 entities × 128 dims
Saved → /Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/TransE_qid_embeddings.pkl


In [79]:
from pykeen.triples import TriplesFactory

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/'
tf = TriplesFactory.from_path(KG_FILES + 'kg_triples.tsv')

print("Relations in graph:")
for rel, idx in sorted(tf.relation_to_id.items(), key=lambda x: x[1]):
    # Count triples using this relation
    mask = tf.mapped_triples[:, 1] == idx
    count = mask.sum().item()
    print(f"  {rel:30} {count:>5} triples")

Relations in graph:
  BORN_IN                          871 triples
  EDUCATED_AT                      725 triples
  HOLDS_POSITION                  2551 triples
  HQ_IN                             32 triples
  LOCATED_IN                       162 triples
  MEMBER_OF                        588 triples
  MEMBER_OF_PARTY                 1272 triples
  REPRESENTS                      1291 triples


In [81]:
import pandas as pd

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/'

df = pd.read_csv(KG_FILES + 'kg_triples.tsv', sep='\t', header=None, names=['subject', 'predicate', 'object'])

print("MEMBER_OF sample:")
print(df[df.predicate == 'MEMBER_OF'][['subject','object']].head(10).to_string())

print("\nMEMBER_OF_PARTY sample:")
print(df[df.predicate == 'MEMBER_OF_PARTY'][['subject','object']].head(10).to_string())

MEMBER_OF sample:
      subject   object
0    Q5046195  Q216517
5      Q76139   Q76382
33    Q154550  Q216517
51    Q974143  Q216517
63   Q6539473  Q216517
69   Q1672642  Q216517
75   Q1410728  Q216517
82    Q454169  Q216517
102   Q380333  Q247135
108  Q6678813  Q216517

MEMBER_OF_PARTY sample:
       subject   object
3     Q5046195  Q216517
19      Q76139   Q76382
26   Q89368401  Q247135
30    Q7782845  Q247135
44     Q154550  Q216517
50  Q106418148    Q9630
59     Q974143  Q216517
67    Q6539473  Q216517
72    Q1672642  Q216517
79    Q1410728  Q216517


In [82]:
member_of = set(zip(df[df.predicate == 'MEMBER_OF']['subject'], 
                    df[df.predicate == 'MEMBER_OF']['object']))

member_of_party = set(zip(df[df.predicate == 'MEMBER_OF_PARTY']['subject'], 
                          df[df.predicate == 'MEMBER_OF_PARTY']['object']))

print(f"MEMBER_OF triples:       {len(member_of)}")
print(f"MEMBER_OF_PARTY triples: {len(member_of_party)}")
print(f"Exact duplicates:        {len(member_of & member_of_party)}")
print(f"Only in MEMBER_OF:       {len(member_of - member_of_party)}")
print(f"Only in MEMBER_OF_PARTY: {len(member_of_party - member_of)}")

MEMBER_OF triples:       588
MEMBER_OF_PARTY triples: 1272
Exact duplicates:        548
Only in MEMBER_OF:       40
Only in MEMBER_OF_PARTY: 724


In [83]:
# Separate out the relations
df_member_of = df[df.predicate == 'MEMBER_OF'].copy()
df_member_of_party = df[df.predicate == 'MEMBER_OF_PARTY'].copy()
df_other = df[~df.predicate.isin(['MEMBER_OF', 'MEMBER_OF_PARTY'])].copy()

# Find the 40 unique MEMBER_OF edges not already in MEMBER_OF_PARTY
party_pairs = set(zip(df_member_of_party['subject'], df_member_of_party['object']))
unique_member_of = df_member_of[
    ~df_member_of.apply(lambda r: (r['subject'], r['object']) in party_pairs, axis=1)
].copy()

# Relabel them as MEMBER_OF_PARTY
unique_member_of['predicate'] = 'MEMBER_OF_PARTY'

# Combine
df_clean = pd.concat([df_other, df_member_of_party, unique_member_of], ignore_index=True)

# Verify
print("Relation counts after merge:")
print(df_clean['predicate'].value_counts().to_string())
print(f"\nTotal triples before: {len(df)}")
print(f"Total triples after:  {len(df_clean)}")

# Save
df_clean.to_csv(KG_FILES + 'kg_triples_clean.tsv', sep='\t', header=False, index=False)
print(f"\nSaved → {KG_FILES}kg_triples_clean.tsv")

Relation counts after merge:
predicate
HOLDS_POSITION     2551
MEMBER_OF_PARTY    1312
REPRESENTS         1291
BORN_IN             871
EDUCATED_AT         725
LOCATED_IN          162
HQ_IN                32

Total triples before: 7492
Total triples after:  6944

Saved → /Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/kg_triples_clean.tsv


In [84]:
import pickle
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

KG_FILES = '/Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/'

tf = TriplesFactory.from_path(KG_FILES + 'kg_triples_clean.tsv')
training, testing, validation = tf.split([0.8, 0.1, 0.1], random_state=42)

print(f"Training triples: {training.num_triples}")
print(f"Validation triples: {validation.num_triples}")
print(f"Test triples: {testing.num_triples}")
print(f"Entities: {tf.num_entities} | Relations: {tf.num_relations}")

result = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='TransE',
    model_kwargs=dict(embedding_dim=128),
    optimizer='Adam',
    optimizer_kwargs=dict(lr=0.001),
    training_kwargs=dict(num_epochs=200, batch_size=256),
    stopper='early',
    stopper_kwargs=dict(frequency=10, patience=5, relative_delta=0.001),
    random_seed=42,
    device='cpu',
)

print(f"\nTraining complete.")
print(f"Hits@10: {result.get_metric('hits@10'):.4f}")
print(f"MRR:     {result.get_metric('mean_reciprocal_rank'):.4f}")

# Extract and save embeddings
entity_embeddings = result.model.entity_representations[0]().detach().cpu().numpy()
entity_to_id = result.training.entity_to_id
id_to_entity = {v: k for k, v in entity_to_id.items()}

qid_to_embedding = {
    id_to_entity[i]: entity_embeddings[i]
    for i in range(len(entity_embeddings))
}

print(f"Embedding dict: {len(qid_to_embedding)} entities × {entity_embeddings.shape[1]} dims")

with open(KG_FILES + 'TransE_qid_embeddings.pkl', 'wb') as f:
    pickle.dump(qid_to_embedding, f)

print(f"Saved → {KG_FILES}TransE_qid_embeddings.pkl")

INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [3579, 694, 695]
INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: /Users/michaelmarkey/.data/pykeen/checkpoints/best-model-weights-dda9bc0d-2cf7-49b5-8ca3-960f7b4b801b.pt
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Training triples: 5555
Validation triples: 695
Test triples: 694
Entities: 2351 | Relations: 7


Training epochs on cpu:   0%|          | 0/200 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.38s seconds
INFO:pykeen.stoppers.early_stopping:Stopping early at epoch 60. The best result 0.1920863309352518 occurred at epoch 10.
INFO:pykeen.stoppers.early_stopping:Re-loading weights from best epoch from /Users/michaelmarkey/.data/pykeen/checkpoints/best-model-weights-dda9bc0d-2cf7-49b5-8ca3-960f7b4b801b.pt


Evaluating on cpu:   0%|          | 0.00/694 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.39s seconds



Training complete.
Hits@10: 0.1981
MRR:     0.0633
Embedding dict: 2351 entities × 128 dims
Saved → /Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/TransE_qid_embeddings.pkl


In [85]:
# Check how many epochs actually ran
print(f"Epochs trained: {len(result.losses)}")
print(f"\nLoss curve (every 10 epochs):")
for i, loss in enumerate(result.losses):
    if (i + 1) % 10 == 0:
        print(f"  Epoch {i+1:>3}: {loss:.4f}")

Epochs trained: 60

Loss curve (every 10 epochs):
  Epoch  10: 0.2090
  Epoch  20: 0.0963
  Epoch  30: 0.0689
  Epoch  40: 0.0488
  Epoch  50: 0.0464
  Epoch  60: 0.0399


In [86]:
result = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='TransE',
    model_kwargs=dict(embedding_dim=128),
    optimizer='Adam',
    optimizer_kwargs=dict(lr=0.001),
    training_kwargs=dict(num_epochs=200, batch_size=256),
    random_seed=42,
    device='cpu',
)

print(f"\nTraining complete.")
print(f"Hits@10: {result.get_metric('hits@10'):.4f}")
print(f"MRR:     {result.get_metric('mean_reciprocal_rank'):.4f}")

print(f"\nEpochs trained: {len(result.losses)}")
print(f"\nLoss curve (every 20 epochs):")
for i, loss in enumerate(result.losses):
    if (i + 1) % 20 == 0:
        print(f"  Epoch {i+1:>3}: {loss:.4f}")

INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Training epochs on cpu:   0%|          | 0/200 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/22.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/694 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.40s seconds



Training complete.
Hits@10: 0.2147
MRR:     0.0757

Epochs trained: 200

Loss curve (every 20 epochs):
  Epoch  20: 0.0963
  Epoch  40: 0.0488
  Epoch  60: 0.0399
  Epoch  80: 0.0392
  Epoch 100: 0.0417
  Epoch 120: 0.0364
  Epoch 140: 0.0400
  Epoch 160: 0.0373
  Epoch 180: 0.0406
  Epoch 200: 0.0384


In [87]:
entity_embeddings = result.model.entity_representations[0]().detach().cpu().numpy()
entity_to_id = result.training.entity_to_id
id_to_entity = {v: k for k, v in entity_to_id.items()}

qid_to_embedding = {
    id_to_entity[i]: entity_embeddings[i]
    for i in range(len(entity_embeddings))
}

with open(KG_FILES + 'TransE_qid_embeddings.pkl', 'wb') as f:
    pickle.dump(qid_to_embedding, f)

print(f"Embedding dict: {len(qid_to_embedding)} entities × {entity_embeddings.shape[1]} dims")
print(f"Saved → {KG_FILES}TransE_qid_embeddings.pkl")

Embedding dict: 2351 entities × 128 dims
Saved → /Users/michaelmarkey/Desktop/Dissertation - 29 05 2026/KG/TransE_qid_embeddings.pkl
